# Multimodal Email & Message Spam Detection Expert: `Qwen2.5-VL-3B` `rsLoRA` & 4-Bit Export

[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-SourinC%2Fqwen2.5--vl--3b--email--sms--spam--rslora--4bit-yellow)](https://huggingface.co/SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit)

**Published Public Model**: [`SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit`](https://huggingface.co/SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit)

## End-to-End Pipeline Overview
1. **Environment-Standard Configuration**: Runs end-to-end locally (dataset ingestion, multimodal rendering, `rsLoRA` training, evaluation, and local `safetensors` / `GGUF` export).
2. **Permissive `Apache-2.0` + `MIT` Public Spam Corpora**: Streams real labeled **Emails** from `Deysi/spam-detection-dataset` (**`Apache-2.0`**) and real labeled **SMS / Text Messages** from `DarkNeuronAI/spam-sms-collection-01` (**`MIT`**), programmatically verifying both open-source licenses via Hugging Face Hub metadata and auditing missing/corrupted records without dropping columns.
3. **Leakage-Free Stratified 80 / 10 / 10 Split, Label-Blind Visual Renderers & EDA**: Deduplicates messages, splits into stratified `Email`/`SMS` $\times$ `SPAM`/`LEGITIMATE` partitions *before* fitting training-only feature scalers, pairs real messages with **label-blind Email Client Screenshots (`PIL`)**, **Mobile SMS Screenshots (`PIL`)**, and **Lexical Risk-Signal Telemetry Charts (`matplotlib`)**, and trains on **both pure Screenshot-Only inputs and Multimodal Image+Text inputs**.
4. **3.75B Multimodal Base Model (`Qwen/Qwen2.5-VL-3B-Instruct`) + Rank-Stabilized QLoRA (`rsLoRA`)**: Fine-tunes the **3.75B-parameter** Vision-Language Model (`>= 3B` requirement) across **both vision and language layers** using **4-bit NormalFloat (`nf4`)** and **Rank-Stabilized LoRA (`rsLoRA`, $r=16, \alpha=16$, scaling updates by $\frac{\alpha}{\sqrt{r}}$)**, evaluating Accuracy, Spam Precision, Spam Recall, F1-Score, and **Side-by-Side Confusion Matrices** against the zero-shot base model.
5. **Dual Compact `<= 3.0 GB` Exports — Standalone 4-Bit `safetensors` (`~2.48 GiB`) & Multimodal `GGUF` (`Q4_K_M` + `mmproj-Q8_0`, `~2.58 GiB`)**: Merges the trained `rsLoRA` adapter in 16-bit and exports both a self-contained **4-bit `NF4` (`bitsandbytes` `safetensors`)** checkpoint (Sections 7–8) and a **`llama.cpp` / LM Studio Multimodal `GGUF`** bundle (`LLM Q4_K_M` + `mmproj-Q8_0`, Section 10)—each staying strictly within the **3.0 GB** budget.

---
## 1. Pipeline Configuration, Optional Environment Token & Runtime Check
> **Prerequisites**: Activate a Python `3.10+` virtual environment (`pip install -r requirements.txt`). No tokens are needed unless you set `PUSH_TO_HF_HUB = True` (which reads `HF_TOKEN` from your environment or `.env`).

In [ ]:
import os
import sys
import math
import getpass
import importlib.util
from pathlib import Path
import torch

# ==============================================================================
# 1. MINIMAL CREDENTIAL RESOLUTION (Zero Credentials Needed for Local Execution)
# ==============================================================================
def load_optional_env_token(var_name: str = "HF_TOKEN", fallback_names=("HUGGINGFACE_TOKEN",)) -> str:
    """Reads an optional token from OS environment variables or a local git-ignored .env file."""
    for key in (var_name, *fallback_names):
        val = os.environ.get(key, "").strip()
        if val:
            return val
    env_file = Path(".env")
    if env_file.is_file():
        for raw_line in env_file.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() in (var_name, *fallback_names):
                cleaned = v.strip().strip("'\"")
                if cleaned:
                    return cleaned
    return ""

HF_TOKEN = load_optional_env_token("HF_TOKEN")
AUTO_INSTALL_MISSING_PKGS = False  # Keep False to use your active virtualenv (pip install -r requirements.txt)

# ==============================================================================
# 2. PUBLIC APACHE-2.0 / MIT SPAM CORPORA & >=3B MULTIMODAL rsLoRA SETTINGS
# ==============================================================================
PUBLISHED_PUBLIC_MODEL_ID = "SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit"
EMAIL_SPAM_DATASET_ID = "Deysi/spam-detection-dataset"         # License: Apache-2.0 (Emails)
SMS_SPAM_DATASET_ID = "DarkNeuronAI/spam-sms-collection-01"    # License: MIT (SMS / Text Messages)
ALLOWED_OPEN_LICENSES = ("apache-2.0", "mit")

EMAIL_STREAM_LIMIT = 1500         # Raw emails streamed for deduplication, audit & balanced curation
SMS_STREAM_LIMIT = 1500           # Raw SMS messages streamed for deduplication, audit & balanced curation
CURATED_SAMPLES_PER_STRATUM = 150 # 4 strata (Email/SMS x Spam/Legit) * 150 = 600 unique messages -> 1,200 multimodal pairs

# Multimodal Base Model: 3.75B Logical Parameters (>= 3B requirement)
BASE_VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
UNSLOTH_VLM_MODEL_ID = "unsloth/Qwen2.5-VL-3B-Instruct"

# Weight Precision & <= 3.0 GB Final Size Budget
# 4-bit NF4 double-quant yields ~2.48 GiB standalone model (upload, download & VRAM <= 3.0 GB).
WEIGHT_PRECISION_BITS = 4         # Options: 4 (recommended for <= 3.0 GB target) or 8
TARGET_MAX_MODEL_SIZE_GB = 3.0

# Rank-Stabilized LoRA (rsLoRA) Hyperparameters
LORA_RANK = 16
LORA_ALPHA = 16
USE_RSLORA = True                 # Scales low-rank updates by alpha / sqrt(r) instead of alpha / r
MAX_TRAIN_STEPS = 70              # Increase or switch to num_train_epochs=1 for full-corpus production runs
LEARNING_RATE = 2e-4

# ==============================================================================
# 3. OPTIONAL PUBLIC HUGGING FACE HUB EXPORT SETTINGS (private=False)
# ==============================================================================
PUSH_TO_HF_HUB = False            # Set True (+ export HF_TOKEN) to publish your own trained weights to HF Hub
HF_NAMESPACE_OVERRIDE = ""        # Leave "" to auto-detect your HF username from HF_TOKEN when publishing
MODEL_REPO_SUFFIX = f"qwen2.5-vl-3b-email-sms-spam-rslora-{WEIGHT_PRECISION_BITS}bit"
DATASET_REPO_SUFFIX = "email-sms-spam-multimodal-curated"
MAKE_HF_REPO_PUBLIC = True        # Enforces PUBLIC visibility (private=False) on Hugging Face Hub
PRIMARY_EXPORT_FORMAT = "safetensors_bnb_4bit"  # Self-contained HF safetensors + AutoProcessor (~2.48 GiB)
EXPORT_AND_PUSH_GGUF = False      # Optional inline flag in Section 7; Section 10 provides full standalone GGUF export
ALSO_PUSH_CURATED_DATASET = False # Optional: set True with PUSH_TO_HF_HUB to publish rendered multimodal splits

# Local Scratch Workspace Directories (Git-Ignored)
STORAGE_ROOT = os.path.abspath("workspace_artifacts")
VISUALS_DIR = os.path.join(STORAGE_ROOT, "multimodal_spam_assets")
SPLITS_DIR = os.path.join(STORAGE_ROOT, "curated_splits")
CHECKPOINTS_DIR = os.path.join(STORAGE_ROOT, "training_checkpoints")
PEFT_ADAPTER_DIR = os.path.join(STORAGE_ROOT, "peft_rslora_spam_adapter")
STANDALONE_MODEL_DIR = os.path.join(STORAGE_ROOT, f"merged_qwen25_vl_3b_spam_rslora_{WEIGHT_PRECISION_BITS}bit")
GGUF_EXPORT_DIR = os.path.join(STORAGE_ROOT, "gguf_qwen25_vl_3b_spam_rslora")
for _sub in ("email_screenshots", "sms_screenshots", "risk_charts"):
    os.makedirs(os.path.join(VISUALS_DIR, _sub), exist_ok=True)
for _dir in (SPLITS_DIR, CHECKPOINTS_DIR, PEFT_ADAPTER_DIR, STANDALONE_MODEL_DIR):
    os.makedirs(_dir, exist_ok=True)

# ==============================================================================
# 4. RUNTIME, PRECISION BUDGET & PACKAGE DISCOVERY SUMMARY (Never Prints Secrets)
# ==============================================================================
rslora_scale = LORA_ALPHA / math.sqrt(LORA_RANK) if USE_RSLORA else LORA_ALPHA / LORA_RANK
print(f"Python Version       : {sys.version.split()[0]}")
print(f"Workspace Root       : {STORAGE_ROOT}")
print(f"Published Reference  : https://huggingface.co/{PUBLISHED_PUBLIC_MODEL_ID}")
print(f"Base VLM             : {BASE_VLM_MODEL_ID} (~3.75B parameters >= 3B)")
print(f"Adapter Technique    : {'rsLoRA' if USE_RSLORA else 'LoRA'} (r={LORA_RANK}, alpha={LORA_ALPHA}, scale={rslora_scale:.2f})")
print(f"Target Export Format : {PRIMARY_EXPORT_FORMAT} ({WEIGHT_PRECISION_BITS}-bit, target <= {TARGET_MAX_MODEL_SIZE_GB:.1f} GB)")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device           : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB VRAM)")
else:
    print("GPU Notice           : No CUDA GPU detected. Sections 1-4 run on CPU; an NVIDIA CUDA GPU is required from Section 5 onward.")

print("\nOptional Hub Push Status (Masked):")
print(f"  - PUSH_TO_HF_HUB   : {PUSH_TO_HF_HUB}")
print(f"  - HF_TOKEN         : {'Detected in environment/.env' if HF_TOKEN else 'Not set (not needed unless PUSH_TO_HF_HUB=True)'}")

required_pkgs = [
    "datasets", "huggingface_hub", "PIL", "matplotlib", "pandas",
    "sklearn", "peft", "accelerate", "trl", "bitsandbytes"
]
print("\nPackage Check:")
for pkg in required_pkgs:
    found = importlib.util.find_spec(pkg.split(".")[0]) is not None
    print(f"  - {pkg:16s}: {'Installed' if found else 'Missing (run: pip install -r requirements.txt)'}")

### Runtime, Precision Budget & Multimodal Spam Architecture Notes
- **Why Multimodal (`Qwen2.5-VL-3B-Instruct`, 3.75B) for Spam & Smishing Detection**: Attackers increasingly evade text-only filters via **image-based spam**, **forwarded mobile SMS / chat screenshots**, and **rendered phishing email captures**, while end users routinely upload screenshots asking *"Is this message spam?"*. Training both vision and language towers equips one unified model to classify **pure screenshots (zero OCR text provided)** as well as **hybrid visual-telemetry + raw-text inputs**.
- **Why 4-Bit (`NF4`) Safetensors Is Pushed by Default (`~2.48 GiB <= 3.0 GB`)**: `Qwen/Qwen2.5-VL-3B-Instruct` has **3.75B parameters** (`~7.00 GiB` in FP16 and `~3.85 GiB` in 8-bit INT8, both exceeding 3 GB). Merging our `rsLoRA` updates and serializing to **standalone 4-bit `NF4` (`bitsandbytes` `safetensors`)** keeps upload, download, and GPU runtime VRAM at **~2.48 GiB**, while keeping vision and language weights in one standard `transformers` repository instead of splitting into two `llama.cpp` GGUF files (`model.gguf` + `mmproj.gguf`).

---
## 2. Verify `Apache-2.0` / `MIT` Licenses & Load Public Email + SMS Spam Corpora
We programmatically verify via `HfApi().dataset_info()` that **`Deysi/spam-detection-dataset`** (`Apache-2.0`) and **`DarkNeuronAI/spam-sms-collection-01`** (`MIT`) carry permissive open-source licenses, stream raw Email and SMS records, and audit schema completeness, missing values, and anomalous targets.

In [ ]:
import subprocess
for base_pkg, mod_name in [
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("scikit-learn", "sklearn"),
    ("pillow", "PIL"),
]:
    if importlib.util.find_spec(mod_name) is None:
        if globals().get("AUTO_INSTALL_MISSING_PKGS", False):
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", base_pkg])
        else:
            raise ImportError(f"Missing '{base_pkg}'. Install dependencies first: pip install -r requirements.txt")

import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

# 1. Programmatically verify permissive MIT / Apache-2.0 licenses on Hugging Face Hub
hub_api = HfApi(token=HF_TOKEN or None)

def verify_permissive_hf_license(dataset_repo_id: str) -> str:
    info = hub_api.dataset_info(dataset_repo_id)
    lic = ""
    if getattr(info, "card_data", None) and getattr(info.card_data, "license", None):
        lic = str(info.card_data.license).lower().strip()
    if not lic:
        for tag in (info.tags or []):
            if tag.startswith("license:"):
                lic = tag.split("license:", 1)[1].lower().strip()
                break
    assert lic in ALLOWED_OPEN_LICENSES, (
        f"Dataset '{dataset_repo_id}' has license '{lic}', expected one of {ALLOWED_OPEN_LICENSES}!"
    )
    return lic

email_lic = verify_permissive_hf_license(EMAIL_SPAM_DATASET_ID)
sms_lic = verify_permissive_hf_license(SMS_SPAM_DATASET_ID)
print(f"Verified Email Corpus : https://huggingface.co/datasets/{EMAIL_SPAM_DATASET_ID} (license='{email_lic}')")
print(f"Verified SMS Corpus   : https://huggingface.co/datasets/{SMS_SPAM_DATASET_ID} (license='{sms_lic}')\n")

# 2. Stream / load raw Email and SMS records with automatic UTF-8 -> CP1252 / Latin-1 fallback
#    (SMS Spam Collection CSVs such as DarkNeuronAI/spam-sms-collection-01 contain Latin-1/CP1252
#     bytes like '£' 0xA3 and occasional unquoted trailing commas that break strict UTF-8 Arrow readers.)
def _recover_ragged_sms_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    unnamed_cols = [c for c in df_in.columns if str(c).lower().startswith("unnamed:")]
    if not unnamed_cols:
        return df_in
    df_fixed = df_in.copy()
    col_map = {str(c).lower().strip(): c for c in df_fixed.columns}
    msg_col = next((col_map[k] for k in ("v2", "message", "text", "sms", "content") if k in col_map), None)
    if msg_col is not None:
        def _join_overflow(row):
            parts = [str(row[msg_col]) if pd.notna(row[msg_col]) else ""]
            for uc in unnamed_cols:
                if pd.notna(row[uc]) and str(row[uc]).strip():
                    parts.append(str(row[uc]).strip())
            return ", ".join(p for p in parts if p)
        df_fixed[msg_col] = df_fixed.apply(_join_overflow, axis=1)
    return df_fixed.drop(columns=unnamed_cols)

def stream_raw_split(repo_id: str, limit: int) -> pd.DataFrame:
    for stream_kwargs in ({}, {"encoding": "cp1252"}, {"encoding": "latin-1"}):
        try:
            ds_stream = load_dataset(
                repo_id, split="train", streaming=True, token=HF_TOKEN or None, **stream_kwargs
            )
            rows = []
            for idx, rec in enumerate(ds_stream):
                if idx >= limit:
                    break
                rows.append(rec)
            if rows:
                return _recover_ragged_sms_columns(pd.DataFrame(rows))
        except Exception:
            continue

    # Direct file fallback via hf_hub_download for non-UTF-8 or ragged CSV/TSV/JSONL files
    repo_files = hub_api.list_repo_files(repo_id=repo_id, repo_type="dataset")
    data_exts = (".parquet", ".csv", ".tsv", ".jsonl", ".json")
    candidates = [
        f for f in repo_files
        if f.lower().endswith(data_exts) and not os.path.basename(f).startswith(".")
    ]
    if not candidates:
        raise FileNotFoundError(f"No tabular data files found in {repo_id}: {repo_files}")
    train_matches = [f for f in candidates if "train" in f.lower() or "spam" in f.lower()]
    target_file = train_matches[0] if train_matches else candidates[0]
    local_path = hf_hub_download(
        repo_id=repo_id, filename=target_file, repo_type="dataset", token=HF_TOKEN or None
    )
    lower_file = target_file.lower()
    if lower_file.endswith(".parquet"):
        return pd.read_parquet(local_path).head(limit)
    if lower_file.endswith((".jsonl", ".json")):
        for enc in ("utf-8", "cp1252", "latin-1"):
            try:
                return pd.read_json(local_path, lines=lower_file.endswith(".jsonl"), encoding=enc).head(limit)
            except UnicodeDecodeError:
                continue
    sep = "\t" if lower_file.endswith(".tsv") else ","
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            df_loaded = pd.read_csv(local_path, sep=sep, encoding=enc, on_bad_lines="skip", nrows=limit)
            if df_loaded.shape[1] == 1 and sep == ",":
                df_tab = pd.read_csv(local_path, sep="\t", encoding=enc, on_bad_lines="skip", nrows=limit)
                if df_tab.shape[1] >= 2:
                    df_loaded = df_tab
            return _recover_ragged_sms_columns(df_loaded)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Unable to decode dataset file '{target_file}' from '{repo_id}'.")

df_email_raw = stream_raw_split(EMAIL_SPAM_DATASET_ID, EMAIL_STREAM_LIMIT)
df_sms_raw = stream_raw_split(SMS_SPAM_DATASET_ID, SMS_STREAM_LIMIT)

print(f"Email Raw Shape : {df_email_raw.shape} | Columns: {list(df_email_raw.columns)}")
print(f"SMS Raw Shape   : {df_sms_raw.shape} | Columns: {list(df_sms_raw.columns)}")

# Resolve text & label columns defensively while keeping original columns intact
def resolve_text_label_columns(df_in: pd.DataFrame) -> tuple:
    col_map = {str(c).lower().strip(): c for c in df_in.columns}
    t_col = next(
        (col_map[k] for k in ("text", "message", "messages", "sms", "sms_text", "body", "email", "email_text", "v2", "content") if k in col_map),
        None,
    )
    l_col = next(
        (col_map[k] for k in ("label", "labels", "category", "spam", "is_spam", "target", "class", "type", "v1") if k in col_map),
        None,
    )
    if (t_col is None or l_col is None) and len(df_in.columns) >= 2:
        valid_cols = [c for c in df_in.columns if not str(c).lower().startswith("unnamed:")]
        if len(valid_cols) >= 2:
            c0, c1 = valid_cols[0], valid_cols[1]
            if df_in[c0].nunique(dropna=True) <= 5 and df_in[c1].nunique(dropna=True) > 5:
                l_col, t_col = l_col or c0, t_col or c1
            elif df_in[c1].nunique(dropna=True) <= 5 and df_in[c0].nunique(dropna=True) > 5:
                t_col, l_col = t_col or c0, l_col or c1
    if t_col is None or l_col is None:
        raise ValueError(f"Could not resolve text/label columns from {list(df_in.columns)}")
    return t_col, l_col

email_tcol, email_lcol = resolve_text_label_columns(df_email_raw)
sms_tcol, sms_lcol = resolve_text_label_columns(df_sms_raw)

df_email_audit = df_email_raw.copy()
df_email_audit["channel"] = "Email"
df_email_audit["source_repo"] = EMAIL_SPAM_DATASET_ID
df_email_audit["raw_message"] = df_email_audit[email_tcol]
df_email_audit["raw_label"] = df_email_audit[email_lcol]

df_sms_audit = df_sms_raw.copy()
df_sms_audit["channel"] = "SMS"
df_sms_audit["source_repo"] = SMS_SPAM_DATASET_ID
df_sms_audit["raw_message"] = df_sms_audit[sms_tcol]
df_sms_audit["raw_label"] = df_sms_audit[sms_lcol]

df_combined_raw = pd.concat(
    [df_email_audit[["channel", "source_repo", "raw_message", "raw_label"]],
     df_sms_audit[["channel", "source_repo", "raw_message", "raw_label"]]],
    ignore_index=True,
)

# 3. Audit Missing, Empty, Too-Short, and Unmapped-Target Rows
def map_spam_target(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in ("spam", "1", "true", "phishing", "malicious", "scam"):
        return 1
    if s in ("not_spam", "not spam", "ham", "0", "false", "legitimate", "benign"):
        return 0
    return np.nan

df_combined_raw["target_id"] = df_combined_raw["raw_label"].apply(map_spam_target)
df_combined_raw["is_message_missing"] = (
    df_combined_raw["raw_message"].isnull()
    | (df_combined_raw["raw_message"].fillna("").astype(str).str.strip() == "")
)
df_combined_raw["is_very_short_anomaly"] = (
    (~df_combined_raw["is_message_missing"])
    & (df_combined_raw["raw_message"].fillna("").astype(str).str.strip().str.len() < 6)
)

audit_table = pd.DataFrame({
    "null_count": df_combined_raw[["channel", "source_repo", "raw_message", "raw_label", "target_id"]].isnull().sum(),
    "empty_str_count": [
        0, 0,
        int((df_combined_raw["raw_message"].fillna("").astype(str).str.strip() == "").sum() - df_combined_raw["raw_message"].isnull().sum()),
        int((df_combined_raw["raw_label"].fillna("").astype(str).str.strip() == "").sum() - df_combined_raw["raw_label"].isnull().sum()),
        0,
    ],
})
audit_table["pct_missing (%)"] = ((audit_table["null_count"] + audit_table["empty_str_count"]) / len(df_combined_raw) * 100).round(2)
print("\nCombined Schema Missing / Null Audit:")
print(audit_table)
print(f"Rows without valid target value : {int(df_combined_raw['target_id'].isnull().sum())}")
print(f"Anomalous ultra-short rows (<6c): {int(df_combined_raw['is_very_short_anomaly'].sum())}")
if df_combined_raw["is_very_short_anomaly"].any():
    print("Sample ultra-short rows:", df_combined_raw[df_combined_raw["is_very_short_anomaly"]][["channel", "raw_label", "raw_message"]].head(3).to_dict(orient="records"))

print("\nRaw Class Distribution by Channel:")
print(pd.crosstab(df_combined_raw["channel"], df_combined_raw["raw_label"].astype(str)))

### Open-Source License Audit & Data Quality Strategy
- **Dual Permissive Open-Source Corpora Verified**: `Deysi/spam-detection-dataset` (`apache-2.0`) covers multi-line corporate and consumer **Email** spam/phishing, while `DarkNeuronAI/spam-sms-collection-01` (`mit`) covers short-form **SMS / Mobile Messaging** smishing, prize lures, and legitimate conversational `ham`.
- **Missing-Value & Anomaly Handling**: All raw columns are preserved. Rows missing a valid target label (`target_id` is `NaN`), empty messages, or ultra-short corrupted fragments (`< 6` characters) are explicitly counted and filtered before deduplication, while missing `Subject:` headers inside raw emails are represented via a binary indicator feature `has_explicit_subject` (`0` imputed as `"[No Explicit Subject]"`).

---
## 3. Deduplication, Stratified 80 / 10 / 10 Split (Strict Featurization Ordering) & EDA
To eliminate data leakage:
1. We normalize and deduplicate message bodies so identical bulk-spam campaigns never appear in both `train` and `test`.
2. We perform a **stratified `Train (80%)` / `Validation (10%)` / `Test (10%)` split** across all 4 `channel × class` strata **before** fitting our 6-feature lexical risk-signal scaler (`95th-percentile` caps fit **strictly on `train_msg_df`**).
3. We visualize class balance, word-count distributions, and training-set risk-signal separation.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# 1. Drop rows without target or valid message text, extract email subject indicator, and deduplicate
df_clean = df_combined_raw[
    (~df_combined_raw["target_id"].isnull())
    & (~df_combined_raw["is_message_missing"])
    & (~df_combined_raw["is_very_short_anomaly"])
].copy()
df_clean["target_id"] = df_clean["target_id"].astype(int)
df_clean["label_name"] = np.where(df_clean["target_id"] == 1, "SPAM", "LEGITIMATE")
df_clean["message_clean"] = df_clean["raw_message"].astype(str).str.strip()

# Feature representing whether an explicit Subject: header exists (imputed cleanly without dropping data)
def parse_subject_and_body(row):
    txt = row["message_clean"]
    if row["channel"] == "Email":
        m = re.match(r"^subject\s*:\s*(.+?)(?:\r?\n|$)(.*)", txt, flags=re.IGNORECASE | re.DOTALL)
        if m:
            subj = m.group(1).strip()[:90] or "[Empty Subject Header]"
            body = m.group(2).strip() or subj
            return pd.Series([True, subj, body])
        return pd.Series([False, "[No Explicit Subject]", txt])
    return pd.Series([False, "[SMS / Mobile Message]", txt])

df_clean[["has_explicit_subject", "subject_display", "body_display"]] = df_clean.apply(parse_subject_and_body, axis=1)
df_clean["dedup_key"] = df_clean["message_clean"].str.lower().str.replace(r"\s+", " ", regex=True)
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["dedup_key"]).reset_index(drop=True)
print(f"Valid labeled records: {before_dedup:,} -> Deduplicated unique messages: {len(df_clean):,}")

# Balance across all 4 strata (Email_LEGITIMATE, Email_SPAM, SMS_LEGITIMATE, SMS_SPAM)
df_clean["stratum"] = df_clean["channel"] + "_" + df_clean["label_name"]
balanced_parts = []
for strat_name, grp in df_clean.groupby("stratum"):
    take_n = min(len(grp), CURATED_SAMPLES_PER_STRATUM)
    balanced_parts.append(grp.sample(n=take_n, random_state=42))
df_curated = pd.concat(balanced_parts, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
df_curated["msg_id"] = [f"msg_{i:05d}" for i in range(len(df_curated))]

# 2. STRICT FEATURIZATION ORDERING: Split into Train (80%) / Val (10%) / Test (10%) FIRST!
train_msg_df, temp_msg_df = train_test_split(
    df_curated, test_size=0.20, random_state=42, stratify=df_curated["stratum"]
)
val_msg_df, test_msg_df = train_test_split(
    temp_msg_df, test_size=0.50, random_state=42, stratify=temp_msg_df["stratum"]
)
train_msg_df = train_msg_df.reset_index(drop=True)
val_msg_df = val_msg_df.reset_index(drop=True)
test_msg_df = test_msg_df.reset_index(drop=True)

assert set(train_msg_df["dedup_key"]).isdisjoint(set(val_msg_df["dedup_key"]))
assert set(train_msg_df["dedup_key"]).isdisjoint(set(test_msg_df["dedup_key"]))
print(f"Unique Message Splits -> Train: {len(train_msg_df)} | Validation: {len(val_msg_df)} | Test: {len(test_msg_df)}")
print("\nStratum Balance Across Splits:")
print(pd.DataFrame({
    "Train": train_msg_df["stratum"].value_counts(),
    "Validation": val_msg_df["stratum"].value_counts(),
    "Test": test_msg_df["stratum"].value_counts(),
}))

# 3. Extract Label-Blind Lexical & Structural Spam Signals (Scaler Fit STRICTLY on train_msg_df!)
RISK_FEATURE_COLS = [
    "url_link_hits",
    "urgency_security_hits",
    "financial_prize_hits",
    "cta_contact_hits",
    "caps_emphasis_ratio",
    "digit_symbol_ratio",
]

def compute_raw_lexical_signals(df_part: pd.DataFrame) -> pd.DataFrame:
    out = df_part.copy()
    texts = out["message_clean"].astype(str)
    out["word_count"] = texts.apply(lambda t: max(1, len(t.split())))
    out["url_link_hits"] = texts.apply(
        lambda t: len(re.findall(r"(https?://|www\.|bit\.ly|tinyurl|\.com/|\.net/|\.org/|click here|http\b)", t, flags=re.I))
    )
    out["urgency_security_hits"] = texts.apply(
        lambda t: len(re.findall(r"\b(urgent|immediately|verify|suspended|account|password|security|expire|limited time|act now|alert|confirm)\b", t, flags=re.I))
    )
    out["financial_prize_hits"] = texts.apply(
        lambda t: len(re.findall(r"(\$|£|€|\bfree\b|\bwinner\b|\bwon\b|\bprize\b|\bcash\b|\bclaim\b|\bcredit\b|\bloan\b|\bguaranteed\b|\bdiscount\b|\bmillion\b|\bbtc\b)", t, flags=re.I))
    )
    out["cta_contact_hits"] = texts.apply(
        lambda t: len(re.findall(r"(\bcall\b|\btxt\b|\breply\b|\bunsubscribe\b|\bopt out\b|\bstop\b|\bclick\b|\b\d{5,11}\b)", t, flags=re.I))
    )
    out["caps_emphasis_ratio"] = texts.apply(
        lambda t: (sum(1 for c in t if c.isupper()) + 2 * t.count("!")) / max(1, len(t))
    )
    out["digit_symbol_ratio"] = texts.apply(
        lambda t: sum(1 for c in t if c.isdigit() or c in "$£€%*#") / max(1, len(t))
    )
    return out

train_msg_df = compute_raw_lexical_signals(train_msg_df)
val_msg_df = compute_raw_lexical_signals(val_msg_df)
test_msg_df = compute_raw_lexical_signals(test_msg_df)

# Fit 95th-percentile normalization caps ONLY on train_msg_df, then transform all splits independently
train_p95_caps = {
    col: max(float(np.percentile(train_msg_df[col], 95)), 1e-3)
    for col in RISK_FEATURE_COLS
}
for part in (train_msg_df, val_msg_df, test_msg_df):
    for col in RISK_FEATURE_COLS:
        part[f"{col}_score100"] = (part[col] / train_p95_caps[col] * 100.0).clip(0.0, 100.0).round(1)

# 4. Render 3-Panel Quantitative Exploratory Data Analysis Figure (Training Split)
fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.3))

ct_split = pd.crosstab(train_msg_df["channel"], train_msg_df["label_name"])[["LEGITIMATE", "SPAM"]]
x_pos = np.arange(len(ct_split.index))
axes[0].bar(x_pos - 0.18, ct_split["LEGITIMATE"], width=0.35, color="#1f77b4", edgecolor="black", label="LEGITIMATE (HAM)")
axes[0].bar(x_pos + 0.18, ct_split["SPAM"], width=0.35, color="#d62728", edgecolor="black", label="SPAM")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(ct_split.index, fontsize=10.5, fontweight="bold")
axes[0].set_title("Training Split Balance by Channel & Verdict", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Unique Messages")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

axes[1].hist(
    train_msg_df[train_msg_df["label_name"] == "LEGITIMATE"]["word_count"].clip(upper=300),
    bins=26, color="#1f77b4", alpha=0.62, edgecolor="black", label="LEGITIMATE (clipped @ 300w)"
)
axes[1].hist(
    train_msg_df[train_msg_df["label_name"] == "SPAM"]["word_count"].clip(upper=300),
    bins=26, color="#d62728", alpha=0.62, edgecolor="black", label="SPAM (clipped @ 300w)"
)
axes[1].set_title("Message Word-Count Distribution (Train Split)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

score_cols = [f"{c}_score100" for c in RISK_FEATURE_COLS]
pretty_names = ["URL / Link", "Urgency / Alert", "Prize / Money", "CTA / Phone", "Caps & '!'", "Digits / Symbols"]
legit_means = train_msg_df[train_msg_df["label_name"] == "LEGITIMATE"][score_cols].mean().values
spam_means = train_msg_df[train_msg_df["label_name"] == "SPAM"][score_cols].mean().values
y_idx = np.arange(len(pretty_names))
axes[2].barh(y_idx - 0.18, legit_means, height=0.35, color="#1f77b4", edgecolor="black", label="LEGITIMATE Mean")
axes[2].barh(y_idx + 0.18, spam_means, height=0.35, color="#d62728", edgecolor="black", label="SPAM Mean")
axes[2].set_yticks(y_idx)
axes[2].set_yticklabels(pretty_names, fontsize=9.5)
axes[2].set_xlabel("Normalized Signal Intensity (0 - 100)")
axes[2].set_title("Label-Blind Risk Signal Separation (Train Split)", fontsize=12, fontweight="bold")
axes[2].legend(loc="lower right")
axes[2].grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

### Exploratory Data Analysis & Feature Separation Insights
- **Bimodal Channel Lengths**: SMS messages cluster tightly below `35` words (short-code lures, OTP spoofs, prize claims), whereas Email traffic spans `40` to `300+` words (newsletter spoofs, phishing account resets, and legitimate business correspondence).
- **Strong Discriminative Signal Without Label Leakage**: Across the training split, `SPAM` exhibits substantially higher `Prize / Money`, `CTA / Phone`, `URL / Link`, `Digits / Symbols`, and `Caps & '!'` intensities than `LEGITIMATE` messages—confirming that our label-blind telemetry chart provides genuine, non-leaky visual signal.

---
## 4. Label-Blind Dual Multimodal Asset Renderers (Email/SMS Screenshots + Risk Charts)
Just like the dual visual modalities in our reference pipeline—and with **zero synthetic LLM calls**—we pair every real open-source Email and SMS message with **two label-blind visual representations**:
1. **Realistic Client Screenshot (`PIL`)**: Neutral desktop Email viewer (for `Email`) or neutral smartphone SMS bubble UI (for `SMS`), using **identical colors and chrome for both `SPAM` and `LEGITIMATE`** so the VLM must inspect actual text, URLs, and layout cues.
2. **Lexical Risk-Signal Telemetry Chart (`matplotlib`)**: 6-axis horizontal bar chart of the input's normalized risk indicators.

Half of screenshot examples are marked `prompt_mode = "screenshot_only"` (omitting raw text from the prompt) so the fine-tuned VLM masters **pure visual screenshot spam detection** alongside multimodal fusion!

In [ ]:
from PIL import Image, ImageDraw, ImageFont

_PIL_CHAR_MAP = str.maketrans({
    "€": "EUR ",
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
    "—": "-",
    "–": "-",
    "…": "...",
    "\u00a0": " ",
})

def safe_pil_text(text: str) -> str:
    s = str(text).translate(_PIL_CHAR_MAP)
    s = re.sub(r"[\x00-\x1f\x7f]", " ", s)
    return s.encode("latin-1", errors="replace").decode("latin-1")

def wrap_text_lines(text: str, max_chars: int = 92, max_lines: int = 14) -> list:
    cleaned = re.sub(r"\s+", " ", safe_pil_text(text)).strip()
    words = cleaned.split(" ")
    lines, cur = [], ""
    for w in words:
        if len(w) > max_chars:
            w = w[:max_chars - 3] + "..."
        if len(cur) + len(w) + 1 <= max_chars:
            cur = f"{cur} {w}".strip()
        else:
            lines.append(cur)
            cur = w
            if len(lines) >= max_lines:
                break
    if cur and len(lines) < max_lines:
        lines.append(cur)
    return lines

# 1. Label-Blind Desktop Email Client Screenshot Renderer (Identical Chrome for SPAM & LEGITIMATE)
def render_email_client_screenshot(row: pd.Series, out_path: str) -> str:
    width, height = 780, 470
    img = Image.new("RGB", (width, height), color=(248, 250, 252))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()

    draw.rectangle([(0, 0), (width, 48)], fill=(30, 41, 59))
    draw.text((18, 16), f"INBOX MESSAGE VIEWER   |   ID: {row['msg_id']}   |   CHANNEL: EMAIL", fill=(241, 245, 249), font=font)

    draw.rectangle([(16, 62), (width - 16, 132)], fill=(255, 255, 255), outline=(203, 213, 225), width=2)
    draw.text((28, 74), f"SUBJECT : {safe_pil_text(row['subject_display'])[:88]}", fill=(15, 23, 42), font=font)
    links_found = re.findall(r"(https?://\S+|www\.\S+)", str(row["message_clean"]), flags=re.I)
    link_summary = safe_pil_text(links_found[0])[:72] if links_found else "None embedded in plain text"
    draw.text((28, 96), f"LINKS   : {link_summary}  (total link tokens={int(row['url_link_hits'])})", fill=(71, 85, 105), font=font)
    draw.text((28, 114), f"LENGTH  : {int(row['word_count'])} words   |   Explicit Subject Header: {bool(row['has_explicit_subject'])}", fill=(100, 116, 139), font=font)

    draw.rectangle([(16, 144), (width - 16, height - 16)], fill=(255, 255, 255), outline=(203, 213, 225), width=1)
    body_lines = wrap_text_lines(row["body_display"], max_chars=96, max_lines=16)
    for idx_l, line in enumerate(body_lines):
        draw.text((28, 158 + idx_l * 18), line, fill=(30, 41, 59), font=font)

    img.save(out_path)
    return out_path

# 2. Label-Blind Mobile SMS / Messaging Screenshot Renderer (Identical Chrome for SPAM & LEGITIMATE)
def render_sms_mobile_screenshot(row: pd.Series, out_path: str) -> str:
    width, height = 460, 470
    img = Image.new("RGB", (width, height), color=(241, 245, 249))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()

    draw.rectangle([(0, 0), (width, 52)], fill=(15, 23, 42))
    draw.text((16, 12), "MOBILE MESSAGING  |  INCOMING SMS / TEXT", fill=(255, 255, 255), font=font)
    draw.text((16, 30), f"Message ID: {row['msg_id']}  |  Words: {int(row['word_count'])}", fill=(148, 163, 184), font=font)

    bubble_lines = wrap_text_lines(row["body_display"], max_chars=52, max_lines=13)
    bubble_h = max(110, 38 + len(bubble_lines) * 18)
    draw.rectangle([(20, 74), (width - 24, 74 + bubble_h)], fill=(255, 255, 255), outline=(148, 163, 184), width=2)
    draw.rectangle([(20, 74), (26, 74 + bubble_h)], fill=(71, 85, 105))
    for idx_l, line in enumerate(bubble_lines):
        draw.text((36, 90 + idx_l * 18), line, fill=(15, 23, 42), font=font)

    footer_y = min(height - 56, 92 + bubble_h)
    draw.rectangle([(20, footer_y), (width - 24, footer_y + 38)], fill=(226, 232, 240), outline=(203, 213, 225), width=1)
    draw.text((30, footer_y + 12), f"Detected Links: {int(row['url_link_hits'])}  |  Phone/CTA Hooks: {int(row['cta_contact_hits'])}", fill=(51, 65, 85), font=font)

    img.save(out_path)
    return out_path

# 3. Label-Blind Lexical Risk-Signal Telemetry Bar Chart Renderer (matplotlib -> PNG)
def render_spam_telemetry_chart(row: pd.Series, out_path: str) -> str:
    labels = ["URL / Links", "Urgency / Alert", "Prize / Money", "CTA / Phone", "Caps & '!'", "Digits / Sym"]
    vals = [
        float(row["url_link_hits_score100"]),
        float(row["urgency_security_hits_score100"]),
        float(row["financial_prize_hits_score100"]),
        float(row["cta_contact_hits_score100"]),
        float(row["caps_emphasis_ratio_score100"]),
        float(row["digit_symbol_ratio_score100"]),
    ]
    fig, ax = plt.subplots(figsize=(6.0, 3.7), dpi=120)
    bar_palette = ["#2563eb", "#7c3aed", "#d97706", "#0891b2", "#4f46e5", "#475569"]
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, vals, color=bar_palette, edgecolor="black", height=0.58)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlim(0, 112)
    ax.set_xlabel("Normalized Input Signal Score (0 - 100)", fontsize=8.5)
    ax.set_title(f"{row['channel']} Inspection Telemetry ({row['msg_id']})", fontsize=10, fontweight="bold")
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    for b in bars:
        w = b.get_width()
        ax.annotate(f"{w:.0f}", xy=(w, b.get_y() + b.get_height() / 2), xytext=(3, 0),
                    textcoords="offset points", ha="left", va="center", fontsize=7.5)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)
    return out_path

# 4. Deterministic Ground-Truth Expert Response Formatter (Grounded Strictly in Dataset Labels)
def build_expert_spam_response(row: pd.Series) -> str:
    active_signals = []
    if row["financial_prize_hits"] > 0:
        active_signals.append("financial/prize lure terms")
    if row["url_link_hits"] > 0:
        active_signals.append("embedded external link/URL cues")
    if row["urgency_security_hits"] > 0:
        active_signals.append("urgent account/verification pressure")
    if row["cta_contact_hits"] > 0:
        active_signals.append("call/reply/click action hook")
    if row["caps_emphasis_ratio_score100"] >= 50:
        active_signals.append("elevated uppercase/exclamation emphasis")
    if row["digit_symbol_ratio_score100"] >= 50:
        active_signals.append("high numeric/code/currency density")

    if int(row["target_id"]) == 1:
        if row["urgency_security_hits"] > 0 and row["url_link_hits"] > 0:
            vector = "Phishing / Deceptive Account Lure"
        elif row["financial_prize_hits"] > 0:
            vector = "Financial / Prize / Promotional Scam"
        elif row["cta_contact_hits"] > 0:
            vector = "Unsolicited Call-to-Action / Smishing"
        else:
            vector = "Unsolicited Bulk Commercial Spam"
        sig_str = ", ".join(active_signals[:3]) if active_signals else "unsolicited promotional phrasing"
        return (
            f"Verdict: SPAM\n"
            f"Channel: {row['channel']}\n"
            f"Threat Vector: {vector}\n"
            f"Analysis: Classified as SPAM based on {sig_str}. Quarantine message and do not click links, call unknown numbers, or disclose credentials."
        )
    return (
        f"Verdict: LEGITIMATE\n"
        f"Channel: {row['channel']}\n"
        f"Threat Vector: None Detected (Normal {row['channel']} Communication)\n"
        f"Analysis: Classified as LEGITIMATE (HAM). Natural conversational or routine correspondence without deceptive spam/smishing hooks."
    )

# 5. Build Dual Multimodal Split Tables (Screenshot Pair + Telemetry Pair per Unique Message)
def materialize_multimodal_split(msg_split_df: pd.DataFrame, split_tag: str) -> pd.DataFrame:
    records = []
    for idx, row in msg_split_df.iterrows():
        expert_ans = build_expert_spam_response(row)
        if row["channel"] == "Email":
            scr_path = render_email_client_screenshot(
                row, os.path.join(VISUALS_DIR, "email_screenshots", f"{split_tag}_{row['msg_id']}.png")
            )
            scr_mod = "email_screenshot"
        else:
            scr_path = render_sms_mobile_screenshot(
                row, os.path.join(VISUALS_DIR, "sms_screenshots", f"{split_tag}_{row['msg_id']}.png")
            )
            scr_mod = "sms_screenshot"

        chart_path = render_spam_telemetry_chart(
            row, os.path.join(VISUALS_DIR, "risk_charts", f"{split_tag}_{row['msg_id']}_chart.png")
        )

        # 50% of screenshot samples train pure visual screenshot spam detection (no raw text in prompt!)
        scr_prompt_mode = "screenshot_only" if (idx % 2 == 0) else "multimodal_image_and_text"

        records.append({
            "sample_id": f"{row['msg_id']}_screen",
            "msg_id": row["msg_id"],
            "channel": row["channel"],
            "source_repo": row["source_repo"],
            "modality_type": scr_mod,
            "prompt_mode": scr_prompt_mode,
            "target_id": int(row["target_id"]),
            "label_name": row["label_name"],
            "message_clean": row["message_clean"],
            "assistant_response": expert_ans,
            "image_path": scr_path,
        })
        records.append({
            "sample_id": f"{row['msg_id']}_telemetry",
            "msg_id": row["msg_id"],
            "channel": row["channel"],
            "source_repo": row["source_repo"],
            "modality_type": "risk_telemetry_chart",
            "prompt_mode": "multimodal_image_and_text",
            "target_id": int(row["target_id"]),
            "label_name": row["label_name"],
            "message_clean": row["message_clean"],
            "assistant_response": expert_ans,
            "image_path": chart_path,
        })
    return pd.DataFrame(records)

print("Rendering label-blind Email screenshots, SMS screenshots, and Risk Telemetry charts across splits...")
train_df = materialize_multimodal_split(train_msg_df, "train")
val_df = materialize_multimodal_split(val_msg_df, "val")
test_df = materialize_multimodal_split(test_msg_df, "test")

train_df.to_parquet(os.path.join(SPLITS_DIR, "train_multimodal.parquet"), index=False)
val_df.to_parquet(os.path.join(SPLITS_DIR, "val_multimodal.parquet"), index=False)
test_df.to_parquet(os.path.join(SPLITS_DIR, "test_multimodal.parquet"), index=False)

print(f"Multimodal Pairs Ready -> Train: {len(train_df):,} | Validation: {len(val_df):,} | Test: {len(test_df):,}")

# Preview all 3 rendered visual modalities side-by-side
email_demo_path = train_df[train_df["modality_type"] == "email_screenshot"].iloc[0]["image_path"]
sms_demo_path = train_df[train_df["modality_type"] == "sms_screenshot"].iloc[0]["image_path"]
chart_demo_path = train_df[train_df["modality_type"] == "risk_telemetry_chart"].iloc[0]["image_path"]

fig_prev, prev_axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax_p, p_path, p_title in [
    (prev_axes[0], email_demo_path, "Modality A: Neutral Desktop Email Screenshot"),
    (prev_axes[1], sms_demo_path, "Modality B: Neutral Mobile SMS Screenshot"),
    (prev_axes[2], chart_demo_path, "Modality C: Input Risk-Signal Telemetry Chart"),
]:
    ax_p.imshow(Image.open(p_path))
    ax_p.set_title(p_title, fontsize=11.5, fontweight="bold")
    ax_p.axis("off")
plt.tight_layout()
plt.show()

### Multimodal Asset Integrity & Zero-Shortcut Verification
- **100% Label-Blind Visual Chrome**: Both `SPAM` and `LEGITIMATE` emails share identical slate-blue headers and white cards, and both `SPAM` and `LEGITIMATE` SMS messages share identical mobile bubbles—preventing visual shortcut learning.
- **Pure Screenshot + Hybrid Coverage**: Training on `screenshot_only` examples forces the VLM's Vision Transformer (`visual`) to read embedded URLs, phone numbers, and scam phrasing directly from pixel patches.

---
## 5. Load `Qwen/Qwen2.5-VL-3B-Instruct` (`3.75B` Params) with 4-Bit `NF4` & Rank-Stabilized LoRA (`rsLoRA`)
> **Hardware Requirement**: Requires an **NVIDIA CUDA GPU** (`>= 12–16 GB` VRAM recommended for `rsLoRA` training and 16-bit merge, e.g., RTX 3060/4070+, L4, A10G, or T4).

This cell verifies VLM dependencies (`trl`, `bitsandbytes`, `peft`, `accelerate`, `qwen-vl-utils`, plus optional `unsloth`), loads the **3.75B-parameter** multimodal base model in `WEIGHT_PRECISION_BITS`-bit precision, attaches **Rank-Stabilized LoRA (`rsLoRA`, `use_rslora=True`, $r=16, \alpha=16$)** across **both vision and language projection layers**, and programmatically asserts `logical_base_params >= 3_000_000_000`.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "An NVIDIA CUDA GPU is required for multimodal rsLoRA fine-tuning! "
        "Verify PyTorch CUDA support with `torch.cuda.is_available()`."
    )

core_ft_pkgs = [
    ("trl>=0.14.0", "trl"),
    ("bitsandbytes>=0.45.0", "bitsandbytes"),
    ("peft>=0.14.0", "peft"),
    ("accelerate>=1.2.0", "accelerate"),
    ("qwen-vl-utils", "qwen_vl_utils"),
]
for pip_spec, mod_name in core_ft_pkgs:
    if importlib.util.find_spec(mod_name) is None:
        if globals().get("AUTO_INSTALL_MISSING_PKGS", False):
            print(f"Installing {pip_spec}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pip_spec])
        else:
            raise ImportError(f"Missing '{pip_spec}'. Run: pip install -r requirements.txt")

assert WEIGHT_PRECISION_BITS in (4, 8), "WEIGHT_PRECISION_BITS must be 4 or 8!"
assert USE_RSLORA is True, "USE_RSLORA must be True to enable Rank-Stabilized LoRA!"

# Primary: Unsloth FastVisionModel (if installed) | Automatic Standard Path: Native Hugging Face 4-bit/8-bit + PEFT rsLoRA
USING_UNSLOTH = False
if WEIGHT_PRECISION_BITS == 4:
    try:
        if importlib.util.find_spec("unsloth") is None and globals().get("AUTO_INSTALL_MISSING_PKGS", False):
            print("Installing unsloth for accelerated Vision rsLoRA...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "unsloth"])
        if importlib.util.find_spec("unsloth") is not None:
            from unsloth import FastVisionModel
            USING_UNSLOTH = True
    except Exception as unsloth_err:
        print(f"Unsloth not active ({unsloth_err}); using Native Hugging Face transformers + PEFT rsLoRA.")

def hw_bf16_supported() -> bool:
    return torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

compute_dtype = torch.bfloat16 if hw_bf16_supported() else torch.float16

# Reuse already-loaded PEFT VLM if re-running this cell in the same GPU session
if "model" in globals() and "processor" in globals() and hasattr(model, "peft_config"):
    print("Reusing already-initialized rsLoRA VLM & processor in GPU memory...")
elif USING_UNSLOTH:
    print(f"[Unsloth Backend] Loading '{UNSLOTH_VLM_MODEL_ID}' in 4-bit NF4 + rsLoRA...")
    model, processor = FastVisionModel.from_pretrained(
        model_name=UNSLOTH_VLM_MODEL_ID,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.0,
        bias="none",
        random_state=42,
        use_rslora=USE_RSLORA,
    )
else:
    import inspect as _ins
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    print(f"[Native HF Backend] Loading '{BASE_VLM_MODEL_ID}' in {WEIGHT_PRECISION_BITS}-bit + rsLoRA...")
    if WEIGHT_PRECISION_BITS == 4:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )
    else:
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_skip_modules=["visual", "lm_head", "merger", "multi_modal_projector"],
        )

    processor = AutoProcessor.from_pretrained(
        BASE_VLM_MODEL_ID,
        min_pixels=256 * 28 * 28,
        max_pixels=640 * 28 * 28,
        token=HF_TOKEN or None,
    )
    processor.tokenizer.padding_side = "right"
    _fp_sig = _ins.signature(Qwen2_5_VLForConditionalGeneration.from_pretrained).parameters
    _dtype_kw = {"dtype": compute_dtype} if "dtype" in _fp_sig else {"torch_dtype": compute_dtype}
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        BASE_VLM_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        **_dtype_kw,
        token=HF_TOKEN or None,
    )
    base_model = prepare_model_for_kbit_training(base_model)
    peft_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        use_rslora=USE_RSLORA,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, peft_config)

# Unpack 4-bit QuantState.shape (compatible across bitsandbytes / PEFT / Transformers 5.x) to verify >= 3.0B params
def count_logical_model_params(peft_vlm):
    if hasattr(peft_vlm, "get_nb_trainable_parameters"):
        try:
            tr_p, all_p = peft_vlm.get_nb_trainable_parameters()
            if all_p >= 3_000_000_000:
                return int(tr_p), int(all_p - tr_p)
        except Exception:
            pass
    trainable_p, logical_base_p = 0, 0
    for _, param in peft_vlm.named_parameters():
        if param.requires_grad:
            trainable_p += int(param.numel())
        else:
            q_state = getattr(param, "quant_state", None)
            q_shape = getattr(q_state, "shape", None) if q_state is not None else None
            if q_shape is not None:
                logical_base_p += int(math.prod(q_shape))
            elif param.__class__.__name__ in ("Params4bit", "Int4Params"):
                logical_base_p += int(param.numel()) * 2
            else:
                logical_base_p += int(param.numel())
    return trainable_p, logical_base_p

trainable_params, logical_base_params = count_logical_model_params(model)
total_logical_params = trainable_params + logical_base_params
assert logical_base_params >= 3_000_000_000, (
    f"Base model must be >= 3B parameters, found {logical_base_params:,}!"
)

print(f"Base VLM Logical Params : {logical_base_params:,} ({logical_base_params / 1e9:.2f}B >= 3.00B verified)")
print(f"rsLoRA Trainable Params : {trainable_params:,} / {total_logical_params:,} ({trainable_params / total_logical_params * 100:.2f}%)")
print(f"rsLoRA Scaling Factor   : alpha / sqrt(r) = {LORA_ALPHA} / sqrt({LORA_RANK}) = {LORA_ALPHA / math.sqrt(LORA_RANK):.2f}")
print(f"Allocated GPU VRAM      : {torch.cuda.memory_allocated(0) / (1024**3):.2f} GiB")

### Model Architecture & Rank-Stabilized LoRA (`rsLoRA`) Parameter Efficiency
- **Verified `3.75B` Parameter Base (`>= 3B`)**: Unpacking `bitsandbytes` `quant_state` shapes confirms **3.75B logical parameters** (`~3.09B` language decoder + `~0.66B` native-resolution Vision Transformer) occupying just **~2.5–2.8 GiB** of VRAM during 4-bit `NF4` loading.
- **Why `rsLoRA` (`use_rslora=True`)**: Standard LoRA scales updates by $\frac{\alpha}{r}$, which shrinks effective adapter learning rates at higher ranks. **Rank-Stabilized LoRA (`rsLoRA`)** scales updates by:
$$\Delta W = \frac{\alpha}{\sqrt{r}} B A$$
At $r=16, \alpha=16$, `rsLoRA` yields a stabilized scaling factor of $4.0$, improving cross-modal alignment between visual screenshot tokens and spam classification heads.

---
## 6. Baseline Evaluation, Multimodal `SFTTrainer` Fine-Tuning, Loss Curve & Confusion Matrices
In spam detection, **False Positives** (flagging legitimate `HAM` emails/SMS as `SPAM`) and **False Negatives** (missing phishing/smishing attacks) have distinct operational costs. Below, we:
1. Evaluate the **Zero-Shot Pre-Trained Base VLM** on a balanced slice of unseen held-out `test` Email & SMS examples (including pure Screenshot-Only cases).
2. Fine-tune with `SFTTrainer` (`rsLoRA`) and plot the **Training & Validation Cross-Entropy Loss Curve**.
3. Re-evaluate the **`rsLoRA` Spam-Expert VLM** on the exact same held-out test set, reporting **Accuracy, Spam Precision, Spam Recall, False Positive Rate, F1-Score**, and **Side-by-Side Confusion Matrices**.

In [ ]:
import re
import inspect
import json
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from trl import SFTTrainer, SFTConfig

# Restore multimodal splits from local Parquet if starting a fresh GPU session
if "train_df" not in globals() or "val_df" not in globals() or "test_df" not in globals():
    train_df = pd.read_parquet(os.path.join(SPLITS_DIR, "train_multimodal.parquet"))
    val_df = pd.read_parquet(os.path.join(SPLITS_DIR, "val_multimodal.parquet"))
    test_df = pd.read_parquet(os.path.join(SPLITS_DIR, "test_multimodal.parquet"))

SPAM_SYSTEM_PROMPT = (
    "You are an Enterprise Email & Messaging Anti-Spam Security Expert. "
    "Inspect the attached visual artifact (email screenshot, mobile SMS screenshot, or risk telemetry chart) "
    "and any provided text. Respond in the exact structured format:\n"
    "Verdict: SPAM or LEGITIMATE\n"
    "Channel: Email or SMS\n"
    "Threat Vector: <concise threat type or None Detected>\n"
    "Analysis: <1-2 sentence security rationale>"
)

def row_to_spam_vlm_conversation(row) -> dict:
    pil_image = Image.open(row["image_path"]).convert("RGB")
    if row["prompt_mode"] == "screenshot_only":
        user_text = (
            f"{SPAM_SYSTEM_PROMPT}\n\n"
            f"[Inspection Mode]: Visual Screenshot Only (read message content directly from image)"
        )
    else:
        user_text = (
            f"{SPAM_SYSTEM_PROMPT}\n\n"
            f"[Channel]: {row['channel']}\n"
            f"[Raw Message Content]:\n{str(row['message_clean'])[:1200]}"
        )
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": pil_image},
                    {"type": "text", "text": user_text},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": str(row["assistant_response"])},
                ],
            },
        ]
    }

train_conversations = [row_to_spam_vlm_conversation(r) for _, r in train_df.iterrows()]
val_conversations = [row_to_spam_vlm_conversation(r) for _, r in val_df.iterrows()]

# Build a stratified 32-sample Held-Out Test Evaluation Benchmark (balanced across Email/SMS x SPAM/LEGITIMATE)
eval_bench_df = (
    test_df.groupby(["channel", "label_name"], as_index=False)
    .head(8)
    .reset_index(drop=True)
)
eval_bench_conversations = [row_to_spam_vlm_conversation(r) for _, r in eval_bench_df.iterrows()]
y_true = eval_bench_df["target_id"].astype(int).tolist()

def predict_spam_verdict(vlm_model, vlm_proc, conv_item: dict) -> tuple:
    img = conv_item["messages"][0]["content"][0]["image"]
    prompt_msgs = [conv_item["messages"][0]]
    chat_text = vlm_proc.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    try:
        inputs = vlm_proc(images=img, text=chat_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    except Exception:
        inputs = vlm_proc(text=[chat_text], images=[img], return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        gen_ids = vlm_model.generate(**inputs, max_new_tokens=110, use_cache=True, do_sample=False)
    raw_out = vlm_proc.batch_decode(gen_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

    first_line = raw_out.split("\n")[0].lower()
    m = re.search(r"verdict\s*:\s*(spam|legitimate|ham|not[_\s]?spam)", raw_out, flags=re.I)
    if m:
        tok = m.group(1).lower()
        pred_label = 1 if tok == "spam" else 0
    elif "not spam" in first_line or "legitimate" in first_line or "ham" in first_line:
        pred_label = 0
    elif "spam" in first_line or "phishing" in first_line or "scam" in first_line:
        pred_label = 1
    else:
        pred_label = 0
    return pred_label, raw_out

# 1. Zero-Shot Pre-Trained Base Model Evaluation
if USING_UNSLOTH:
    FastVisionModel.for_inference(model)
else:
    model.eval()

print(f"Evaluating Zero-Shot Pre-Trained Base VLM on {len(eval_bench_conversations)} held-out Email & SMS test cases...")
base_results = [predict_spam_verdict(model, processor, ex) for ex in eval_bench_conversations]
y_pred_base = [r[0] for r in base_results]

# 2. Multimodal rsLoRA Fine-Tuning with SFTTrainer (Version-Safe for TRL & Transformers 4.x / 5.x)
sft_init_params = inspect.signature(SFTTrainer.__init__).parameters
if USING_UNSLOTH:
    try:
        from unsloth.trainer import UnslothVisionDataCollator
    except ImportError:
        from unsloth import UnslothVisionDataCollator
    FastVisionModel.for_training(model)
    vision_collator = UnslothVisionDataCollator(model, processor)
    trainer_proc_kw = {"processing_class": processor} if "processing_class" in sft_init_params else {"tokenizer": processor}
else:
    model.train()
    assistant_header_ids = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def native_qwen_vl_collator(examples):
        texts = [processor.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False) for ex in examples]
        images = [
            [item["image"] for msg in ex["messages"] for item in msg["content"] if item["type"] == "image"]
            for ex in examples
        ]
        batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        for special_tok in ("<|vision_start|>", "<|vision_end|>", "<|image_pad|>"):
            tok_id = processor.tokenizer.convert_tokens_to_ids(special_tok)
            if tok_id is not None and tok_id != processor.tokenizer.unk_token_id:
                labels[labels == tok_id] = -100
        hdr_len = len(assistant_header_ids)
        if hdr_len > 0:
            for row_idx in range(labels.shape[0]):
                seq = batch["input_ids"][row_idx].tolist()
                for pos in range(len(seq) - hdr_len + 1):
                    if seq[pos:pos + hdr_len] == assistant_header_ids:
                        labels[row_idx, :pos + hdr_len] = -100
                        break
        batch["labels"] = labels
        return batch

    vision_collator = native_qwen_vl_collator
    trainer_proc_kw = (
        {"processing_class": processor.tokenizer}
        if "processing_class" in sft_init_params
        else {"tokenizer": processor.tokenizer}
    )

use_hw_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
sft_cfg_params = inspect.signature(SFTConfig.__init__).parameters
extra_sft_cfg = {}
if "dataset_text_field" in sft_cfg_params:
    extra_sft_cfg["dataset_text_field"] = ""
if "dataset_kwargs" in sft_cfg_params:
    extra_sft_cfg["dataset_kwargs"] = {"skip_prepare_dataset": True}
if "max_length" in sft_cfg_params:
    extra_sft_cfg["max_length"] = 2048
elif "max_seq_length" in sft_cfg_params:
    extra_sft_cfg["max_seq_length"] = 2048
eval_strat_key = "eval_strategy" if "eval_strategy" in sft_cfg_params else "evaluation_strategy"
extra_sft_cfg[eval_strat_key] = "steps"

trainer = SFTTrainer(
    model=model,
    data_collator=vision_collator,
    train_dataset=train_conversations,
    eval_dataset=val_conversations,
    **trainer_proc_kw,
    args=SFTConfig(
        per_device_train_batch_size=2 if USING_UNSLOTH else 1,
        gradient_accumulation_steps=4 if USING_UNSLOTH else 8,
        warmup_steps=6,
        max_steps=MAX_TRAIN_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not use_hw_bf16,
        bf16=use_hw_bf16,
        logging_steps=5,
        eval_steps=15,
        save_strategy="steps",
        save_steps=35,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=CHECKPOINTS_DIR,
        report_to="none",
        remove_unused_columns=False,
        **extra_sft_cfg,
    ),
)

print("\nStarting Multimodal Spam-Expert rsLoRA Fine-Tuning...")
train_stats = trainer.train()
print(f"Training Complete! Final Training Loss: {train_stats.training_loss:.4f}")

# Save PEFT rsLoRA adapter & training history locally
model.save_pretrained(PEFT_ADAPTER_DIR)
processor.save_pretrained(PEFT_ADAPTER_DIR)
with open(os.path.join(PEFT_ADAPTER_DIR, "training_log_history.json"), "w", encoding="utf-8") as log_f:
    json.dump(trainer.state.log_history, log_f, indent=2)

# 3. Post-Training Evaluation on Held-Out Test Benchmark
if USING_UNSLOTH:
    FastVisionModel.for_inference(model)
else:
    model.eval()

ft_results = [predict_spam_verdict(model, processor, ex) for ex in eval_bench_conversations]
y_pred_ft = [r[0] for r in ft_results]

def summarize_spam_metrics(y_t, y_p, name: str) -> dict:
    cm = confusion_matrix(y_t, y_p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_t, y_p), 4),
        "Spam Precision": round(precision_score(y_t, y_p, zero_division=0), 4),
        "Spam Recall": round(recall_score(y_t, y_p, zero_division=0), 4),
        "Spam F1": round(f1_score(y_t, y_p, zero_division=0), 4),
        "False Positive Rate": round(fp / max(1, (fp + tn)), 4),
    }

metrics_df = pd.DataFrame([
    summarize_spam_metrics(y_true, y_pred_base, "Zero-Shot Base Qwen2.5-VL-3B"),
    summarize_spam_metrics(y_true, y_pred_ft, "rsLoRA Fine-Tuned Spam Expert"),
])
print("\n=== Held-Out Test Set Spam Classification Metrics ===")
print(metrics_df.to_string(index=False))

# 4. Plot Loss Curve + Side-by-Side Confusion Matrices
log_history = trainer.state.log_history
train_steps = [x["step"] for x in log_history if "loss" in x]
train_losses = [x["loss"] for x in log_history if "loss" in x]
eval_steps = [x["step"] for x in log_history if "eval_loss" in x]
eval_losses = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig_eval, eval_axes = plt.subplots(1, 3, figsize=(18.5, 5.0))
eval_axes[0].plot(train_steps, train_losses, marker="o", color="#1f77b4", label="Training Loss", linewidth=2)
if eval_losses:
    eval_axes[0].plot(eval_steps, eval_losses, marker="s", color="#d62728", label="Validation Loss", linewidth=2)
eval_axes[0].set_title("Multimodal rsLoRA Loss Curve", fontsize=12, fontweight="bold")
eval_axes[0].set_xlabel("Optimization Step")
eval_axes[0].set_ylabel("Cross-Entropy Loss")
eval_axes[0].legend()
eval_axes[0].grid(True, linestyle="--", alpha=0.4)

for ax_cm, preds_cm, title_cm, cmap_name in [
    (eval_axes[1], y_pred_base, "Confusion Matrix: Zero-Shot Base", "Blues"),
    (eval_axes[2], y_pred_ft, "Confusion Matrix: rsLoRA Spam Expert", "Greens"),
]:
    cm_mat = confusion_matrix(y_true, preds_cm, labels=[0, 1])
    im = ax_cm.imshow(cm_mat, cmap=cmap_name)
    ax_cm.set_xticks([0, 1])
    ax_cm.set_yticks([0, 1])
    ax_cm.set_xticklabels(["Pred LEGIT", "Pred SPAM"], fontweight="bold")
    ax_cm.set_yticklabels(["True LEGIT", "True SPAM"], fontweight="bold")
    ax_cm.set_title(title_cm, fontsize=12, fontweight="bold")
    for r_i in range(2):
        for c_i in range(2):
            ax_cm.text(c_i, r_i, str(cm_mat[r_i, c_i]), ha="center", va="center",
                       fontsize=14, fontweight="bold", color="white" if cm_mat[r_i, c_i] > cm_mat.max() / 2 else "black")

plt.tight_layout()
eval_plot_path = os.path.join(PEFT_ADAPTER_DIR, "training_loss_and_confusion_matrices.png")
fig_eval.savefig(eval_plot_path, dpi=140)
plt.show()

# Qualitative Held-Out Test Comparison
print("\n--- Held-Out Test Qualitative Sample (Unseen Message) ---")
print(f"Channel / Mode    : {eval_bench_df.iloc[0]['channel']} ({eval_bench_df.iloc[0]['prompt_mode']})")
print(f"Ground Truth      :\n{eval_bench_df.iloc[0]['assistant_response']}\n")
print(f"rsLoRA Prediction :\n{ft_results[0][1]}")

### Spam Classification Performance & Confusion Matrix Insights
- **Precision vs. False Positive Control**: Evaluating **Spam Precision**, **Spam Recall**, **Spam F1**, and **False Positive Rate** alongside side-by-side confusion matrices verifies that `rsLoRA` fine-tuning sharpens spam detection across both Email and SMS screenshots without over-blocking legitimate `HAM` correspondence.
- **Structured Explainable Verdicts**: Unlike black-box binary classifiers, the fine-tuned VLM outputs a parseable `Verdict:`, `Channel:`, `Threat Vector:`, and `Analysis:` grounded in the visual screenshot and lexical telemetry.

---
## 7. Merge `rsLoRA` Adapter & Export Standalone 4-Bit `NF4` `safetensors` Model (`<= 3.0 GB`)
To guarantee that the **published model's upload size, download size, and GPU runtime VRAM footprint are all `<= 3.0 GB`**:
1. **Primary Export (`safetensors_bnb_4bit`)**: We merge the trained `rsLoRA` weights ($\Delta W = \frac{\alpha}{\sqrt{r}} B A$) into clean 16-bit base weights and serialize a **self-contained 4-bit NormalFloat (`NF4` double-quantized `bitsandbytes` `safetensors`)** directory (`~2.46 GiB / ~2.64 GB`) with `AutoProcessor` and embedded `quantization_config` in `config.json`.
2. **Optional Secondary Export (`EXPORT_AND_PUSH_GGUF = True`)**: Converts the merged 16-bit checkpoint via `llama.cpp` `convert_hf_to_gguf.py` into `GGUF` + `--mmproj`.

In [ ]:
import gc
import inspect
import shutil
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

def get_directory_size_gib(folder: str) -> float:
    total_bytes = 0
    for root, _, files in os.walk(folder):
        for fname in files:
            total_bytes += os.path.getsize(os.path.join(root, fname))
    return total_bytes / (1024 ** 3)

if os.path.exists(STANDALONE_MODEL_DIR):
    shutil.rmtree(STANDALONE_MODEL_DIR)
os.makedirs(STANDALONE_MODEL_DIR, exist_ok=True)

export_completed = False
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
_vlm_fp_sig = inspect.signature(Qwen2_5_VLForConditionalGeneration.from_pretrained).parameters
export_dtype_kw = {"dtype": compute_dtype} if "dtype" in _vlm_fp_sig else {"torch_dtype": compute_dtype}

# Path A: Direct Unsloth 4-bit merged safetensors export when using Unsloth in 4-bit mode and GGUF is not requested
if globals().get("USING_UNSLOTH", False) and "model" in globals() and WEIGHT_PRECISION_BITS == 4 and not EXPORT_AND_PUSH_GGUF:
    try:
        print("[Unsloth Export] Merging rsLoRA weights into standalone 4-bit NF4 safetensors...")
        model.save_pretrained_merged(
            STANDALONE_MODEL_DIR,
            processor,
            save_method="merged_4bit_forced",
        )
        has_safetensors = any(f.endswith(".safetensors") for f in os.listdir(STANDALONE_MODEL_DIR))
        if has_safetensors and get_directory_size_gib(STANDALONE_MODEL_DIR) <= TARGET_MAX_MODEL_SIZE_GB:
            export_completed = True
        elif has_safetensors:
            print(f"Unsloth dynamic 4-bit export was {get_directory_size_gib(STANDALONE_MODEL_DIR):.2f} GiB (> {TARGET_MAX_MODEL_SIZE_GB:.1f} GiB); running uniform NF4 re-quantization...")
    except Exception as unsloth_merge_err:
        print(f"Unsloth direct 4-bit merge fallback triggered ({unsloth_merge_err}); running exact 16-bit merge -> 4-bit quantization.")

# Path B: Universal 2-Stage Exact FP16/BF16 rsLoRA Merge -> Clean 4-Bit/8-Bit Safetensors Serialization
if not export_completed:
    temp_merged_16bit_dir = os.path.join(STORAGE_ROOT, "_temp_merged_16bit")
    if os.path.exists(temp_merged_16bit_dir):
        shutil.rmtree(temp_merged_16bit_dir)
    os.makedirs(temp_merged_16bit_dir, exist_ok=True)

    merged_16bit_ready = False
    if globals().get("USING_UNSLOTH", False) and "model" in globals():
        try:
            print("  1/3 [Unsloth 16-Bit Merge] Streaming lossless 16-bit rsLoRA merge to scratch disk...")
            model.save_pretrained_merged(temp_merged_16bit_dir, processor, save_method="merged_16bit")
            merged_16bit_ready = any(f.endswith(".safetensors") for f in os.listdir(temp_merged_16bit_dir))
        except Exception as u16_err:
            print(f"  Unsloth 16-bit merge fallback ({u16_err}); using PEFT merge_and_unload...")

    print("[2-Stage Exact Merge] Releasing training buffers to free GPU VRAM...")
    for _obj_name in ("trainer", "model", "base_model"):
        if _obj_name in globals():
            del globals()[_obj_name]
    gc.collect()
    torch.cuda.empty_cache()

    processor_export = AutoProcessor.from_pretrained(PEFT_ADAPTER_DIR)
    if not merged_16bit_ready:
        print(f"  1/3 Loading clean 16-bit '{BASE_VLM_MODEL_ID}' and merging rsLoRA spam-expert adapter...")
        base_16bit = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            BASE_VLM_MODEL_ID,
            **export_dtype_kw,
            low_cpu_mem_usage=True,
            device_map="cuda:0",
            token=HF_TOKEN or None,
        )
        merged_16bit = PeftModel.from_pretrained(base_16bit, PEFT_ADAPTER_DIR).merge_and_unload()
        merged_16bit.save_pretrained(temp_merged_16bit_dir, safe_serialization=True, max_shard_size="4GB")
        processor_export.save_pretrained(temp_merged_16bit_dir)
        del merged_16bit, base_16bit
        gc.collect()
        torch.cuda.empty_cache()

    # Optional Multimodal GGUF Export (LLM + --mmproj) from clean 16-bit merged weights
    if EXPORT_AND_PUSH_GGUF:
        os.makedirs(GGUF_EXPORT_DIR, exist_ok=True)
        llama_cpp_dir = os.path.join(STORAGE_ROOT, "llama_cpp_converter")
        if not os.path.exists(os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")):
            subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", llama_cpp_dir])
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gguf", "sentencepiece", "protobuf"])
        conv_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
        llm_gguf_path = os.path.join(GGUF_EXPORT_DIR, "qwen2.5-vl-3b-spam-rslora-q8_0.gguf")
        mmproj_gguf_path = os.path.join(GGUF_EXPORT_DIR, "mmproj-qwen2.5-vl-3b-spam-rslora-q8_0.gguf")
        subprocess.check_call([sys.executable, conv_script, temp_merged_16bit_dir, "--outfile", llm_gguf_path, "--outtype", "q8_0"])
        subprocess.check_call([sys.executable, conv_script, temp_merged_16bit_dir, "--outfile", mmproj_gguf_path, "--outtype", "q8_0", "--mmproj"])
        print(f"  [GGUF Export] Generated GGUF + mmproj in {GGUF_EXPORT_DIR} ({get_directory_size_gib(GGUF_EXPORT_DIR):.2f} GiB)")

    print(f"  2/3 Quantizing merged 3.75B Spam-Expert VLM to standalone {WEIGHT_PRECISION_BITS}-bit safetensors...")
    if WEIGHT_PRECISION_BITS == 4:
        export_bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=["lm_head", "merger", "multi_modal_projector"],
        )
    else:
        export_bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_skip_modules=["visual", "lm_head", "merger", "multi_modal_projector"],
        )

    quantized_standalone = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        temp_merged_16bit_dir,
        quantization_config=export_bnb_config,
        **export_dtype_kw,
        low_cpu_mem_usage=True,
        device_map="cuda:0",
    )
    print(f"  3/3 Saving standalone {WEIGHT_PRECISION_BITS}-bit model + processor to '{STANDALONE_MODEL_DIR}'...")
    if os.path.exists(STANDALONE_MODEL_DIR):
        shutil.rmtree(STANDALONE_MODEL_DIR)
    os.makedirs(STANDALONE_MODEL_DIR, exist_ok=True)
    quantized_standalone.save_pretrained(STANDALONE_MODEL_DIR, safe_serialization=True, max_shard_size="5GB")
    processor_export.save_pretrained(STANDALONE_MODEL_DIR)

    del quantized_standalone
    shutil.rmtree(temp_merged_16bit_dir, ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

# Copy lightweight evaluation plots & logs into standalone model repo directory
for asset_name in ("training_loss_and_confusion_matrices.png", "training_log_history.json"):
    src_asset = os.path.join(PEFT_ADAPTER_DIR, asset_name)
    if os.path.exists(src_asset):
        shutil.copy2(src_asset, os.path.join(STANDALONE_MODEL_DIR, asset_name))
if "metrics_df" in globals():
    metrics_df.to_json(os.path.join(STANDALONE_MODEL_DIR, "test_spam_metrics.json"), orient="records", indent=2)

export_gib = get_directory_size_gib(STANDALONE_MODEL_DIR)
export_gb_decimal = export_gib * (1024 ** 3) / (1000 ** 3)
print(f"\nStandalone Export Directory : {STANDALONE_MODEL_DIR}")
for fname in sorted(os.listdir(STANDALONE_MODEL_DIR)):
    fpath = os.path.join(STANDALONE_MODEL_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  - {fname:42s}: {os.path.getsize(fpath) / (1024**2):8.2f} MiB")

print(f"\nTotal Upload / Download Footprint : {export_gib:.2f} GiB ({export_gb_decimal:.2f} GB)")
if WEIGHT_PRECISION_BITS == 4:
    assert export_gib <= TARGET_MAX_MODEL_SIZE_GB, (
        f"Exported 4-bit model ({export_gib:.2f} GiB) exceeded {TARGET_MAX_MODEL_SIZE_GB:.1f} GiB budget!"
    )
    print(f"Size Budget Verification          : PASSED ({export_gib:.2f} GiB <= {TARGET_MAX_MODEL_SIZE_GB:.1f} GiB target)")

# Visualize Precision & Format Size Comparison vs. 3.0 GB Budget
fig_sz, ax_sz = plt.subplots(figsize=(8.8, 4.3))
format_labels = [
    "Merged FP16 / BF16\n(safetensors)",
    "Merged 8-Bit INT8\n(bitsandbytes)",
    "Multimodal GGUF\n(Q4_K_M + mmproj-Q8_0)",
    f"Our Exported {WEIGHT_PRECISION_BITS}-Bit NF4\n(Merged safetensors)",
]
format_sizes_gib = [7.00, 3.85, 2.78, round(export_gib, 2)]
bar_colors = ["#94a3b8", "#f59e0b", "#38bdf8", "#16a34a"]
bars = ax_sz.bar(format_labels, format_sizes_gib, color=bar_colors, edgecolor="black", width=0.52)
ax_sz.axhline(TARGET_MAX_MODEL_SIZE_GB, color="#dc2626", linestyle="--", linewidth=2, label=f"Max Size Budget ({TARGET_MAX_MODEL_SIZE_GB:.1f} GiB)")
ax_sz.set_title("Qwen2.5-VL-3B Spam-Expert (3.75B) Standalone Export Size by Format", fontsize=12, fontweight="bold")
ax_sz.set_ylabel("Model Repository Size (GiB)")
ax_sz.set_ylim(0, 8.2)
ax_sz.legend()
ax_sz.grid(axis="y", linestyle="--", alpha=0.4)
for b in bars:
    h = b.get_height()
    ax_sz.annotate(f"{h:.2f} GiB", xy=(b.get_x() + b.get_width() / 2, h), xytext=(0, 4),
                   textcoords="offset points", ha="center", va="bottom", fontweight="bold", fontsize=9.5)
plt.tight_layout()
plt.show()

### Standalone 4-Bit `safetensors` Export & `<= 3.0 GB` Budget Verification
- **Unified Vision + Text Serialization**: Unlike text-only GGUF converters that separate `--mmproj` from the language model, our standalone 4-bit `NF4` `safetensors` directory preserves **both** the `rsLoRA`-adapted Vision Transformer and language decoder in a single `~2.46 GiB (~2.64 GB)` checkpoint.
- **Zero Extra Base Download at Runtime**: Because `quantization_config` is embedded directly inside `config.json`, downstream users download only `~2.48 GiB` and allocate `~2.7 GiB` GPU VRAM—never pulling the `7.0 GiB` FP16 base model.

---
## 8. Generate Model Card & Optionally Publish Standalone 4-Bit VLM to Hugging Face Hub (`private=False`)
This cell always writes a complete `README.md` Model Card locally into `STANDALONE_MODEL_DIR`. When **`PUSH_TO_HF_HUB = True`** (with `HF_TOKEN` configured in your environment or `.env`), it creates a **Public (`private=False`)** Hugging Face Model Repository, uploads the standalone `~2.48 GiB` model, and verifies `private == False`.

In [ ]:
import getpass
from datasets import Dataset, DatasetDict, Image as HFImage
from huggingface_hub import HfApi, login

export_gib_val = get_directory_size_gib(STANDALONE_MODEL_DIR)
metrics_md_table = ""
if "metrics_df" in globals():
    try:
        metrics_md_table = metrics_df.to_markdown(index=False)
    except Exception:
        metrics_md_table = "```\n" + metrics_df.to_string(index=False) + "\n```"

hf_username_card = (globals().get("HF_NAMESPACE_OVERRIDE", "") or "").strip() or "<hf-username>"
if PUSH_TO_HF_HUB:
    if not globals().get("HF_TOKEN", ""):
        HF_TOKEN = load_optional_env_token("HF_TOKEN")
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass("Enter your Hugging Face Write Token (input hidden): ").strip()
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN (Write permission) is required when PUSH_TO_HF_HUB=True!")
    login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi(token=HF_TOKEN)
    hf_username_card = (globals().get("HF_NAMESPACE_OVERRIDE", "") or "").strip() or api.whoami()["name"]

MODEL_REPO_ID = f"{hf_username_card}/{MODEL_REPO_SUFFIX}"
is_private_flag = not bool(MAKE_HF_REPO_PUBLIC)  # False -> PUBLIC repository!

# Write Public Model Card (README.md) locally into STANDALONE_MODEL_DIR
model_card_md = f"""---
library_name: transformers
base_model: {BASE_VLM_MODEL_ID}
datasets:
- {EMAIL_SPAM_DATASET_ID}
- {SMS_SPAM_DATASET_ID}
pipeline_tag: image-text-to-text
tags:
- vision-language
- qwen2.5-vl
- multimodal
- spam-detection
- phishing-detection
- smishing
- email-security
- rslora
- qlora
- bitsandbytes
- {WEIGHT_PRECISION_BITS}-bit
license: other
license_name: qwen-research
license_link: https://huggingface.co/{BASE_VLM_MODEL_ID}/blob/main/LICENSE
---

# {MODEL_REPO_ID}

**Built with Qwen** — Standalone **{WEIGHT_PRECISION_BITS}-bit (`NF4` double-quantized `safetensors`)** multimodal **Email & SMS/Message Spam Detection Expert**, fine-tuned from [`{BASE_VLM_MODEL_ID}`](https://huggingface.co/{BASE_VLM_MODEL_ID}) (**3.75B parameters**) using **Rank-Stabilized LoRA (`rsLoRA`, $r={LORA_RANK}, \\alpha={LORA_ALPHA}$)**.

## Capabilities
- **Pure Screenshot Spam & Smishing Detection**: Classify desktop email client screenshots or mobile SMS/chat screenshots directly from pixels without external OCR.
- **Multimodal Hybrid Detection**: Jointly analyze lexical risk-telemetry charts and raw email/message text to output structured `Verdict: SPAM | LEGITIMATE`, `Channel`, `Threat Vector`, and security `Analysis`.
- **Compact `<= 3 GB` Standalone Footprint**: Merged and quantized to **{export_gib_val:.2f} GiB** (`{WEIGHT_PRECISION_BITS}`-bit `NF4` `safetensors` + `AutoProcessor`), loading directly in ~2.7 GiB GPU VRAM without downloading the 7.0 GiB FP16 base model.

## Open-Source Training Corpora (Zero Synthetic LLM Data)
- **Emails**: [`{EMAIL_SPAM_DATASET_ID}`](https://huggingface.co/datasets/{EMAIL_SPAM_DATASET_ID}) (**Apache-2.0** license)
- **SMS / Mobile Messages**: [`{SMS_SPAM_DATASET_ID}`](https://huggingface.co/datasets/{SMS_SPAM_DATASET_ID}) (**MIT** license)

## Held-Out Test Set Evaluation
{metrics_md_table}

## Quickstart (Standalone 4-Bit Multimodal Spam Check)
```python
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

repo_id = "{MODEL_REPO_ID}"
processor = AutoProcessor.from_pretrained(repo_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(repo_id, device_map="auto")

screenshot = Image.open("suspicious_message_screenshot.png").convert("RGB")
prompt_text = (
    "You are an Enterprise Email & Messaging Anti-Spam Security Expert. "
    "Inspect the attached visual artifact (email screenshot, mobile SMS screenshot, or risk telemetry chart) "
    "and any provided text. Respond in the exact structured format:\\n"
    "Verdict: SPAM or LEGITIMATE\\nChannel: Email or SMS\\n"
    "Threat Vector: <concise threat type or None Detected>\\nAnalysis: <1-2 sentence security rationale>\\n\\n"
    "[Inspection Mode]: Visual Screenshot Only (read message content directly from image)"
)
messages = [{{"role": "user", "content": [{{"type": "image", "image": screenshot}}, {{"type": "text", "text": prompt_text}}]}}]
chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[chat], images=[screenshot], return_tensors="pt", padding=True).to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=110, do_sample=False)
print(processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0])
```

## Licensing & Attribution
- Training corpora are licensed under **Apache-2.0** (`{EMAIL_SPAM_DATASET_ID}`) and **MIT** (`{SMS_SPAM_DATASET_ID}`).
- *Qwen is licensed under the Qwen RESEARCH LICENSE AGREEMENT, Copyright (c) Alibaba Cloud. All Rights Reserved.*
"""

with open(os.path.join(STANDALONE_MODEL_DIR, "README.md"), "w", encoding="utf-8") as card_f:
    card_f.write(model_card_md)
print(f"Wrote local Model Card -> {os.path.join(STANDALONE_MODEL_DIR, 'README.md')}")

if PUSH_TO_HF_HUB:
    api.create_repo(repo_id=MODEL_REPO_ID, repo_type="model", private=is_private_flag, exist_ok=True)
    try:
        api.update_repo_visibility(repo_id=MODEL_REPO_ID, repo_type="model", private=is_private_flag)
    except Exception:
        pass

    print(f"Uploading standalone {WEIGHT_PRECISION_BITS}-bit Spam-Expert VLM ({export_gib_val:.2f} GiB) to PUBLIC HF repo: https://huggingface.co/{MODEL_REPO_ID} ...")
    api.upload_folder(
        folder_path=STANDALONE_MODEL_DIR,
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        commit_message=f"Publish standalone {WEIGHT_PRECISION_BITS}-bit rsLoRA Email & SMS Spam-Expert Qwen2.5-VL-3B ({export_gib_val:.2f} GiB)",
        token=HF_TOKEN,
    )
    repo_meta = api.repo_info(repo_id=MODEL_REPO_ID, repo_type="model")
    assert repo_meta.private is is_private_flag, f"Expected private={is_private_flag}, got private={repo_meta.private}!"
    print(f"  -> Model Upload Complete! Verified Public Visibility: public={not repo_meta.private} (private={repo_meta.private})")

    if EXPORT_AND_PUSH_GGUF and os.path.exists(GGUF_EXPORT_DIR) and os.listdir(GGUF_EXPORT_DIR):
        GGUF_REPO_ID = f"{MODEL_REPO_ID}-gguf"
        api.create_repo(repo_id=GGUF_REPO_ID, repo_type="model", private=is_private_flag, exist_ok=True)
        api.upload_folder(folder_path=GGUF_EXPORT_DIR, repo_id=GGUF_REPO_ID, repo_type="model", token=HF_TOKEN)
        print(f"  -> Optional Multimodal GGUF Repo Published: https://huggingface.co/{GGUF_REPO_ID}")

    if ALSO_PUSH_CURATED_DATASET:
        DATASET_REPO_ID = f"{hf_username_card}/{DATASET_REPO_SUFFIX}"
        def _split_to_hf(df_part):
            recs = df_part.rename(columns={"image_path": "image"}).to_dict(orient="list")
            return Dataset.from_dict(recs).cast_column("image", HFImage())
        curated_dd = DatasetDict({"train": _split_to_hf(train_df), "validation": _split_to_hf(val_df), "test": _split_to_hf(test_df)})
        api.create_repo(repo_id=DATASET_REPO_ID, repo_type="dataset", private=is_private_flag, exist_ok=True)
        curated_dd.push_to_hub(DATASET_REPO_ID, token=HF_TOKEN, private=is_private_flag)
        print(f"  -> Optional Curated Multimodal Dataset Published: https://huggingface.co/datasets/{DATASET_REPO_ID}")
else:
    print("Skipping Hugging Face Hub upload (PUSH_TO_HF_HUB=False).")
    print(f"Local standalone {WEIGHT_PRECISION_BITS}-bit model ready at: {STANDALONE_MODEL_DIR}")
    print(f"Published reference model available at: https://huggingface.co/{PUBLISHED_PUBLIC_MODEL_ID}")

### Public Repository & Deployment Readiness
- **Verified Public Visibility (`private=False`)**: Anyone can inspect the Model Card, evaluation metrics, loss/confusion-matrix plots, and pull the standalone `~2.48 GiB` spam detector without an access token.

---
## 9. One-Call Public Reload & Live Email / SMS Spam Inspection Helper
Use `reload_public_spam_expert()` and `inspect_message_or_screenshot()` in any future session or anti-spam service to classify either a **screenshot image alone** or **image + raw message text**.

In [ ]:
def reload_public_spam_expert(model_repo_id: str = "SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit"):
    """Loads the public standalone 4-bit Qwen2.5-VL-3B Spam-Expert (~2.48 GiB download, zero credentials required)."""
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    repo_id = model_repo_id or globals().get("PUBLISHED_PUBLIC_MODEL_ID", "SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit")
    reloaded_proc = AutoProcessor.from_pretrained(repo_id)
    reloaded_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        repo_id,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    reloaded_model.eval()
    vram_gib = torch.cuda.memory_allocated(0) / (1024**3) if torch.cuda.is_available() else 0.0
    print(f"Loaded public Spam-Expert VLM '{repo_id}' | GPU VRAM: {vram_gib:.2f} GiB")
    return reloaded_model, reloaded_proc

def inspect_message_or_screenshot(vlm_model, vlm_proc, image_path_or_pil, raw_text: str = None, channel: str = "Email") -> str:
    """Classifies an email/SMS screenshot (with optional raw text) as SPAM or LEGITIMATE."""
    from PIL import Image
    sys_prompt = globals().get(
        "SPAM_SYSTEM_PROMPT",
        (
            "You are an Enterprise Email & Messaging Anti-Spam Security Expert. "
            "Inspect the attached visual artifact (email screenshot, mobile SMS screenshot, or risk telemetry chart) "
            "and any provided text. Respond in the exact structured format:\n"
            "Verdict: SPAM or LEGITIMATE\nChannel: Email or SMS\n"
            "Threat Vector: <concise threat type or None Detected>\nAnalysis: <1-2 sentence security rationale>"
        ),
    )
    pil_img = image_path_or_pil.convert("RGB") if isinstance(image_path_or_pil, Image.Image) else Image.open(image_path_or_pil).convert("RGB")
    if raw_text and raw_text.strip():
        body_prompt = f"{sys_prompt}\n\n[Channel]: {channel}\n[Raw Message Content]:\n{raw_text.strip()[:1200]}"
    else:
        body_prompt = f"{sys_prompt}\n\n[Inspection Mode]: Visual Screenshot Only (read message content directly from image)"
    msgs = [{"role": "user", "content": [{"type": "image", "image": pil_img}, {"type": "text", "text": body_prompt}]}]
    chat = vlm_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    try:
        inputs = vlm_proc(text=[chat], images=[pil_img], return_tensors="pt", padding=True).to("cuda")
    except Exception:
        inputs = vlm_proc(images=pil_img, text=chat, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_ids = vlm_model.generate(**inputs, max_new_tokens=110, do_sample=False)
    return vlm_proc.batch_decode(out_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

print("Ready! In any session, run:")
print(f"  spam_vlm, spam_proc = reload_public_spam_expert('{globals().get('MODEL_REPO_ID', globals().get('PUBLISHED_PUBLIC_MODEL_ID', 'SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit'))}')")
print("  print(inspect_message_or_screenshot(spam_vlm, spam_proc, 'email_or_sms_capture.png'))")

### Public `safetensors` Reload Readiness
- `reload_public_spam_expert()` and `inspect_message_or_screenshot()` provide zero-credential Python GPU inference directly from [`SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit`](https://huggingface.co/SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit) (or your own local/published checkpoint).

---
## 10. Export Multimodal `GGUF` (`Q4_K_M` + `mmproj-Q8_0`, `<= 3.0 GB`) & Optional Hub Publish
For local CPU/GPU/Metal inference in **`llama.cpp` (`llama-mtmd-cli` / `llama-server`)**, **LM Studio**, or **Ollama**, Vision-Language Models require **two GGUF files**:
1. **Main Language Backbone (`Q4_K_M`)**: `qwen2.5-vl-3b-email-sms-spam-rslora-Q4_K_M.gguf` (**~1.80 GiB / ~1.93 GB**)
2. **Vision Multimodal Projector (`--mmproj`, `Q8_0`)**: `mmproj-qwen2.5-vl-3b-email-sms-spam-rslora-Q8_0.gguf` (**~0.78 GiB / ~0.84 GB**)

Together, both files total **~2.58 GiB (~2.77 GB <= 3.0 GB)** inside `workspace_artifacts/gguf_qwen25_vl_3b_spam_rslora/` and can optionally be published to a **Public (`private=False`)** companion repository (`<hf-username>/qwen2.5-vl-3b-email-sms-spam-rslora-4bit-GGUF`) when `PUSH_GGUF_TO_PUBLIC_HF = True`.

In [ ]:
import os
import sys
import gc
import json
import glob
import shutil
import inspect
import getpass
import subprocess
import importlib.util
import torch
from huggingface_hub import HfApi, login, snapshot_download
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

# ==============================================================================
# 1. MULTIMODAL GGUF EXPORT & OPTIONAL PUBLIC HF PUSH CONFIG (<= 3.0 GB TOTAL)
# ==============================================================================
GGUF_LLM_QUANT_TYPE = "Q4_K_M"      # ~1.80 GiB (~1.93 GB) main language backbone
GGUF_MMPROJ_OUTTYPE = "q8_0"        # ~0.78 GiB (~0.84 GB) vision tower -> ~2.58 GiB (~2.77 GB <= 3.0 GB) combined!
PUSH_GGUF_TO_PUBLIC_HF = bool(globals().get("PUSH_TO_HF_HUB", False))  # Set True (+ HF_TOKEN) to publish to HF Hub

STORAGE_ROOT = globals().get("STORAGE_ROOT", os.path.abspath("workspace_artifacts"))
PEFT_ADAPTER_DIR = globals().get("PEFT_ADAPTER_DIR", os.path.join(STORAGE_ROOT, "peft_rslora_spam_adapter"))
GGUF_EXPORT_DIR = globals().get("GGUF_EXPORT_DIR", os.path.join(STORAGE_ROOT, "gguf_qwen25_vl_3b_spam_rslora"))
BASE_VLM_MODEL_ID = globals().get("BASE_VLM_MODEL_ID", "Qwen/Qwen2.5-VL-3B-Instruct")
PUBLISHED_PUBLIC_MODEL_ID = globals().get("PUBLISHED_PUBLIC_MODEL_ID", "SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit")
MODEL_REPO_SUFFIX = globals().get("MODEL_REPO_SUFFIX", "qwen2.5-vl-3b-email-sms-spam-rslora-4bit")
TARGET_MAX_MODEL_SIZE_GB = globals().get("TARGET_MAX_MODEL_SIZE_GB", 3.0)
MAKE_HF_REPO_PUBLIC = globals().get("MAKE_HF_REPO_PUBLIC", True)
EMAIL_SPAM_DATASET_ID = globals().get("EMAIL_SPAM_DATASET_ID", "Deysi/spam-detection-dataset")
SMS_SPAM_DATASET_ID = globals().get("SMS_SPAM_DATASET_ID", "DarkNeuronAI/spam-sms-collection-01")
LORA_RANK = globals().get("LORA_RANK", 16)
LORA_ALPHA = globals().get("LORA_ALPHA", 16)

if not os.path.exists(PEFT_ADAPTER_DIR) and "model" not in globals():
    raise FileNotFoundError(
        f"Trained rsLoRA adapter not found at '{PEFT_ADAPTER_DIR}'. Run Section 6 first to save the fine-tuned adapter."
    )

if os.path.exists(GGUF_EXPORT_DIR):
    shutil.rmtree(GGUF_EXPORT_DIR)
os.makedirs(GGUF_EXPORT_DIR, exist_ok=True)

# ==============================================================================
# 2. BUILD CLEAN 16-BIT MERGED CHECKPOINT FOR LOSS-FREE GGUF CONVERSION
# ==============================================================================
temp_gguf_16bit_dir = os.path.join(STORAGE_ROOT, "_temp_gguf_merged_16bit")
if os.path.exists(temp_gguf_16bit_dir):
    shutil.rmtree(temp_gguf_16bit_dir)
os.makedirs(temp_gguf_16bit_dir, exist_ok=True)

# Free any reloaded inference models from Section 9 before 16-bit merge
for _stale_var in ("spam_vlm", "reloaded_model"):
    if _stale_var in globals():
        del globals()[_stale_var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

merged_ok = False
if globals().get("USING_UNSLOTH", False) and "model" in globals() and hasattr(model, "save_pretrained_merged"):
    try:
        print("[1/4] Streaming 16-bit rsLoRA merge via Unsloth for GGUF export...")
        model.save_pretrained_merged(temp_gguf_16bit_dir, processor, save_method="merged_16bit")
        merged_ok = any(f.endswith(".safetensors") for f in os.listdir(temp_gguf_16bit_dir))
    except Exception as u_err:
        print(f"  Unsloth live merge fallback ({u_err}); merging saved PEFT adapter onto clean 16-bit base...")

if not merged_ok:
    for _v in ("trainer", "model", "base_model"):
        if _v in globals():
            del globals()[_v]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    compute_dtype = (
        torch.bfloat16
        if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8)
        else torch.float16
    )
    _fp_params = inspect.signature(Qwen2_5_VLForConditionalGeneration.from_pretrained).parameters
    _dtype_kw = {"dtype": compute_dtype} if "dtype" in _fp_params else {"torch_dtype": compute_dtype}

    try:
        print(f"[1/4] Loading clean 16-bit '{BASE_VLM_MODEL_ID}' and merging rsLoRA adapter from '{PEFT_ADAPTER_DIR}'...")
        base_16 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            BASE_VLM_MODEL_ID,
            **_dtype_kw,
            low_cpu_mem_usage=True,
            device_map="cuda:0" if torch.cuda.is_available() else "cpu",
            token=globals().get("HF_TOKEN", "") or None,
        )
        merged_16 = PeftModel.from_pretrained(base_16, PEFT_ADAPTER_DIR).merge_and_unload()
        merged_16.save_pretrained(temp_gguf_16bit_dir, safe_serialization=True, max_shard_size="4GB")
        AutoProcessor.from_pretrained(PEFT_ADAPTER_DIR).save_pretrained(temp_gguf_16bit_dir)
        del merged_16, base_16
        merged_ok = True
    except Exception as peft_err:
        print(f"  Native PEFT merge fallback ({peft_err}); reloading via Unsloth FastVisionModel...")
        from unsloth import FastVisionModel
        u_model, u_proc = FastVisionModel.from_pretrained(model_name=PEFT_ADAPTER_DIR, load_in_4bit=True)
        u_model.save_pretrained_merged(temp_gguf_16bit_dir, u_proc, save_method="merged_16bit")
        del u_model, u_proc
        merged_ok = True

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Sync canonical Qwen2.5-VL tokenizer & config metadata so llama.cpp convert_hf_to_gguf.py
# works cleanly across both Transformers 4.x and Transformers 5.x
ref_meta_dir = snapshot_download(
    repo_id=BASE_VLM_MODEL_ID,
    allow_patterns=["vocab.json", "merges.txt", "tokenizer.json", "tokenizer_config.json", "preprocessor_config.json", "chat_template.json", "config.json"],
    token=globals().get("HF_TOKEN", "") or None,
)
for meta_fname in ("vocab.json", "merges.txt", "tokenizer.json", "tokenizer_config.json", "preprocessor_config.json", "chat_template.json"):
    src_meta = os.path.join(ref_meta_dir, meta_fname)
    if os.path.exists(src_meta):
        shutil.copy2(src_meta, os.path.join(temp_gguf_16bit_dir, meta_fname))

ref_cfg_path = os.path.join(ref_meta_dir, "config.json")
cur_cfg_path = os.path.join(temp_gguf_16bit_dir, "config.json")
if os.path.exists(ref_cfg_path) and os.path.exists(cur_cfg_path):
    with open(ref_cfg_path, "r", encoding="utf-8") as f_ref, open(cur_cfg_path, "r", encoding="utf-8") as f_cur:
        ref_cfg, cur_cfg = json.load(f_ref), json.load(f_cur)
    cur_cfg.pop("quantization_config", None)
    for k, v in ref_cfg.items():
        if k not in cur_cfg:
            cur_cfg[k] = v
    with open(cur_cfg_path, "w", encoding="utf-8") as f_out:
        json.dump(cur_cfg, f_out, indent=2)

# ==============================================================================
# 3. SETUP LLAMA.CPP CONVERTER + CPU LLAMA-QUANTIZE BINARY
# ==============================================================================
print("[2/4] Preparing llama.cpp GGUF converter & llama-quantize tool...")
llama_cpp_dir = os.path.join(STORAGE_ROOT, "llama_cpp_gguf")
if not os.path.exists(os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", llama_cpp_dir])

for _mod_name, _pip_pkg in (("sentencepiece", "sentencepiece"), ("google.protobuf", "protobuf")):
    if importlib.util.find_spec(_mod_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pip_pkg])
gguf_py_dir = os.path.join(llama_cpp_dir, "gguf-py")
if os.path.exists(gguf_py_dir):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", gguf_py_dir])
elif importlib.util.find_spec("gguf") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "gguf"])

def resolve_or_build_llama_quantize(repo_dir: str) -> str:
    exe_suffix = ".exe" if os.name == "nt" else ""
    bin_name = f"llama-quantize{exe_suffix}"
    search_paths = [
        shutil.which("llama-quantize"),
        os.path.join(repo_dir, "build", "bin", bin_name),
        os.path.join(repo_dir, "build", "bin", "Release", bin_name),
        os.path.join(repo_dir, bin_name),
        os.path.abspath(os.path.join("llama.cpp", bin_name)),
        os.path.abspath(os.path.join("llama.cpp", "build", "bin", bin_name)),
    ]
    for p in search_paths:
        if p and os.path.isfile(p) and (os.name == "nt" or os.access(p, os.X_OK)):
            return p
    if shutil.which("cmake") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cmake"])
    build_dir = os.path.join(repo_dir, "build")
    subprocess.check_call([
        "cmake", "-S", repo_dir, "-B", build_dir,
        "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DBUILD_SHARED_LIBS=OFF",
    ])
    subprocess.check_call([
        "cmake", "--build", build_dir, "--config", "Release", "--target", "llama-quantize", "-j",
    ])
    built_candidates = glob.glob(os.path.join(build_dir, "**", f"llama-quantize*{exe_suffix}"), recursive=True)
    for cand in built_candidates:
        if os.path.isfile(cand):
            return cand
    raise RuntimeError("Could not locate compiled llama-quantize binary!")

conv_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
quant_bin = resolve_or_build_llama_quantize(llama_cpp_dir)

# ==============================================================================
# 4. EXPORT BOTH MULTIMODAL GGUF FILES (LLM Q4_K_M + MMPROJ Q8_0 <= 3.0 GB)
# ==============================================================================
print(f"[3/4] Converting Language Backbone -> {GGUF_LLM_QUANT_TYPE} and Vision Projector (--mmproj) -> {GGUF_MMPROJ_OUTTYPE.upper()}...")
tmp_llm_f16_gguf = os.path.join(STORAGE_ROOT, "_tmp_qwen25_vl_3b_f16.gguf")
final_llm_gguf_name = f"qwen2.5-vl-3b-email-sms-spam-rslora-{GGUF_LLM_QUANT_TYPE}.gguf"
final_llm_gguf_path = os.path.join(GGUF_EXPORT_DIR, final_llm_gguf_name)

subprocess.check_call([
    sys.executable, conv_script, temp_gguf_16bit_dir,
    "--outfile", tmp_llm_f16_gguf, "--outtype", "f16",
])
subprocess.check_call([quant_bin, tmp_llm_f16_gguf, final_llm_gguf_path, GGUF_LLM_QUANT_TYPE])
if os.path.exists(tmp_llm_f16_gguf):
    os.remove(tmp_llm_f16_gguf)

mmproj_used_type = GGUF_MMPROJ_OUTTYPE.upper()
final_mmproj_name = f"mmproj-qwen2.5-vl-3b-email-sms-spam-rslora-{mmproj_used_type}.gguf"
final_mmproj_path = os.path.join(GGUF_EXPORT_DIR, final_mmproj_name)
try:
    subprocess.check_call([
        sys.executable, conv_script, "--mmproj", temp_gguf_16bit_dir,
        "--outfile", final_mmproj_path, "--outtype", GGUF_MMPROJ_OUTTYPE.lower(),
    ])
except subprocess.CalledProcessError:
    mmproj_used_type = "F16"
    final_mmproj_name = "mmproj-qwen2.5-vl-3b-email-sms-spam-rslora-F16.gguf"
    final_mmproj_path = os.path.join(GGUF_EXPORT_DIR, final_mmproj_name)
    subprocess.check_call([
        sys.executable, conv_script, "--mmproj", temp_gguf_16bit_dir,
        "--outfile", final_mmproj_path, "--outtype", "f16",
    ])

# Remove temporary 16-bit staging directory
shutil.rmtree(temp_gguf_16bit_dir, ignore_errors=True)

gguf_total_bytes = sum(
    os.path.getsize(os.path.join(GGUF_EXPORT_DIR, f))
    for f in os.listdir(GGUF_EXPORT_DIR)
    if os.path.isfile(os.path.join(GGUF_EXPORT_DIR, f))
)
gguf_total_gib = gguf_total_bytes / (1024 ** 3)
gguf_total_gb = gguf_total_bytes / (1000 ** 3)

print(f"\nGGUF Export Directory : {GGUF_EXPORT_DIR}")
for fname in sorted(os.listdir(GGUF_EXPORT_DIR)):
    fp = os.path.join(GGUF_EXPORT_DIR, fname)
    if os.path.isfile(fp):
        print(f"  - {fname:58s}: {os.path.getsize(fp) / (1024**2):8.2f} MiB ({os.path.getsize(fp) / (1024**3):.2f} GiB)")
print(f"Combined Multimodal GGUF Size : {gguf_total_gib:.2f} GiB ({gguf_total_gb:.2f} GB)")

# ==============================================================================
# 5. WRITE GGUF MODEL CARD & OPTIONALLY PUBLISH TO PUBLIC HF REPO (private=False)
# ==============================================================================
hf_user_preview = (globals().get("HF_NAMESPACE_OVERRIDE", "") or "").strip() or "<hf-username>"
GGUF_REPO_ID = f"{hf_user_preview}/{MODEL_REPO_SUFFIX}-GGUF"
SAFETENSORS_REPO_ID = globals().get("MODEL_REPO_ID", PUBLISHED_PUBLIC_MODEL_ID)

if PUSH_GGUF_TO_PUBLIC_HF:
    hf_tok = globals().get("HF_TOKEN", "")
    if not hf_tok and "load_optional_env_token" in globals():
        hf_tok = load_optional_env_token("HF_TOKEN")
    if not hf_tok:
        hf_tok = getpass.getpass("Enter your Hugging Face Write Token for GGUF upload (input hidden): ").strip()

    login(token=hf_tok, add_to_git_credential=False)
    gguf_api = HfApi(token=hf_tok)
    hf_user = (globals().get("HF_NAMESPACE_OVERRIDE", "") or "").strip() or gguf_api.whoami()["name"]
    GGUF_REPO_ID = f"{hf_user}/{MODEL_REPO_SUFFIX}-GGUF"
    SAFETENSORS_REPO_ID = globals().get("MODEL_REPO_ID", f"{hf_user}/{MODEL_REPO_SUFFIX}")

gguf_readme = f"""---
library_name: gguf
base_model: {BASE_VLM_MODEL_ID}
datasets:
- {EMAIL_SPAM_DATASET_ID}
- {SMS_SPAM_DATASET_ID}
pipeline_tag: image-text-to-text
tags:
- gguf
- llama-cpp
- vision-language
- qwen2.5-vl
- multimodal
- spam-detection
- phishing-detection
- smishing
- rslora
license: other
license_name: qwen-research
license_link: https://huggingface.co/{BASE_VLM_MODEL_ID}/blob/main/LICENSE
---

# {GGUF_REPO_ID}

**Built with Qwen** — Standalone **Multimodal `GGUF` (`{GGUF_LLM_QUANT_TYPE}` + `mmproj-{mmproj_used_type}`, {gguf_total_gib:.2f} GiB total)** export of our **Email & SMS/Message Spam Detection Expert**, fine-tuned from [`{BASE_VLM_MODEL_ID}`](https://huggingface.co/{BASE_VLM_MODEL_ID}) (**3.75B parameters**) using **Rank-Stabilized LoRA (`rsLoRA`, $r={LORA_RANK}, \\alpha={LORA_ALPHA}$)**.

- **Companion `transformers` 4-Bit `safetensors` Repo**: [`{SAFETENSORS_REPO_ID}`](https://huggingface.co/{SAFETENSORS_REPO_ID})
- **Open-Source Training Corpora**: [`{EMAIL_SPAM_DATASET_ID}`](https://huggingface.co/datasets/{EMAIL_SPAM_DATASET_ID}) (`Apache-2.0`) + [`{SMS_SPAM_DATASET_ID}`](https://huggingface.co/datasets/{SMS_SPAM_DATASET_ID}) (`MIT`)

## Included Multimodal GGUF Files (`{gguf_total_gib:.2f} GiB` / `{gguf_total_gb:.2f} GB` Combined)
| File | Role | Precision |
| :--- | :--- | :--- |
| `{final_llm_gguf_name}` | Main 3B Language Backbone | `{GGUF_LLM_QUANT_TYPE}` |
| `{final_mmproj_name}` | Vision Encoder & Multimodal Projector (`--mmproj`) | `{mmproj_used_type}` |

> **Important**: Download **both** files above to classify email or SMS screenshots in `llama.cpp` or **LM Studio**.

## Quickstart with `llama.cpp` (`llama-mtmd-cli` / `llama-server`)
```bash
# 1. Single Screenshot Spam Check via llama-mtmd-cli
./llama-mtmd-cli \\
  -m {final_llm_gguf_name} \\
  --mmproj {final_mmproj_name} \\
  --image suspicious_email_or_sms.png \\
  -p "You are an Enterprise Email & Messaging Anti-Spam Security Expert. Inspect the attached visual artifact and respond with Verdict: SPAM or LEGITIMATE, Channel, Threat Vector, and Analysis."

# 2. OpenAI-Compatible Multimodal HTTP Server
./llama-server \\
  -m {final_llm_gguf_name} \\
  --mmproj {final_mmproj_name} \\
  --port 8080
```
"""
with open(os.path.join(GGUF_EXPORT_DIR, "README.md"), "w", encoding="utf-8") as gf:
    gf.write(gguf_readme)

if PUSH_GGUF_TO_PUBLIC_HF:
    gguf_private_flag = not bool(MAKE_HF_REPO_PUBLIC)
    print(f"\n[4/4] Uploading Multimodal GGUF ({gguf_total_gib:.2f} GiB) to PUBLIC HF Repo: https://huggingface.co/{GGUF_REPO_ID} ...")
    gguf_api.create_repo(repo_id=GGUF_REPO_ID, repo_type="model", private=gguf_private_flag, exist_ok=True)
    try:
        gguf_api.update_repo_visibility(repo_id=GGUF_REPO_ID, repo_type="model", private=gguf_private_flag)
    except Exception:
        pass
    gguf_api.upload_folder(
        folder_path=GGUF_EXPORT_DIR,
        repo_id=GGUF_REPO_ID,
        repo_type="model",
        commit_message=f"Publish Multimodal GGUF ({GGUF_LLM_QUANT_TYPE} + mmproj-{mmproj_used_type}, {gguf_total_gib:.2f} GiB) Email & SMS Spam-Expert VLM",
        token=hf_tok,
    )
    gguf_meta = gguf_api.repo_info(repo_id=GGUF_REPO_ID, repo_type="model")
    assert gguf_meta.private is gguf_private_flag, f"Expected private={gguf_private_flag}, got {gguf_meta.private}!"
    print(f"  -> Public GGUF Repo Published : https://huggingface.co/{GGUF_REPO_ID} (public={not gguf_meta.private})")
else:
    print(f"\n[4/4] Local Multimodal GGUF bundle ready in '{GGUF_EXPORT_DIR}' (PUSH_GGUF_TO_PUBLIC_HF=False; skipping Hub upload).")

### Multimodal `GGUF` Architecture & `<= 3.0 GB` Footprint Notes
- **Two-File Vision-Language GGUF Packaging**: Quantizing the 3.09B language backbone to **`Q4_K_M` (~1.80 GiB)** and exporting the 0.66B vision encoder/projector (`--mmproj`) in **`Q8_0` (~0.78 GiB)** keeps the **entire multimodal GGUF bundle at ~2.58 GiB (~2.77 GB <= 3.0 GB)** while preserving sharp OCR/layout reading on mobile SMS and desktop email screenshots.
- **Cross-Platform Edge Deployment**: The exported GGUF bundle runs natively on CPU, Apple Silicon Metal, Vulkan, and CUDA via `llama-mtmd-cli`, `llama-server`, and LM Studio—complementing the single-directory Python `transformers` 4-bit `safetensors` model [`SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit`](https://huggingface.co/SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit).

---
## 11. Final Summary

### Q&A
- **How does this notebook train a multimodal Email & Message Spam Detection Expert without LLM data synthesis or external API keys?**
  It streams real labeled **Emails** from **`Deysi/spam-detection-dataset`** (**`Apache-2.0`** license) and real labeled **SMS / Mobile Messages** from **`DarkNeuronAI/spam-sms-collection-01`** (**`MIT`** license), verifying both permissive licenses programmatically via Hugging Face Hub metadata with zero synthetic text generation or API keys. Each deduplicated message is paired with label-blind **Email / Mobile SMS Client Screenshots** (`PIL`) and **Lexical Risk-Signal Telemetry Charts** (`matplotlib`), training **`Qwen/Qwen2.5-VL-3B-Instruct`** (**3.75B parameters**) via **Rank-Stabilized LoRA (`rsLoRA`, $r=16, \alpha=16$, scaling by $\frac{\alpha}{\sqrt{r}} = 4.0$)** across both vision and language layers to detect spam from **pure screenshots** as well as **hybrid image+text inputs**.
- **What model formats are supported (`safetensors` and `GGUF`), and how is the `3 GB` budget achieved for each?**
  1. **Primary Python / `transformers` 4-Bit `safetensors` (`Section 7–8`)**: Merges `rsLoRA` in 16-bit and exports a **Standalone 4-bit NormalFloat (`NF4` double-quantized `bitsandbytes` `safetensors`)** checkpoint (`~2.46 GiB / ~2.64 GB <= 3.0 GB`), published publicly at [`SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit`](https://huggingface.co/SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit).
  2. **Companion `llama.cpp` / Edge Multimodal `GGUF` (`Section 10`)**: Converts the merged 16-bit model into **`qwen2.5-vl-3b-email-sms-spam-rslora-Q4_K_M.gguf` (~1.80 GiB)** plus **`mmproj-qwen2.5-vl-3b-email-sms-spam-rslora-Q8_0.gguf` (~0.78 GiB)**—a combined **`~2.58 GiB (~2.77 GB <= 3.0 GB)`** multimodal GGUF bundle for local or Hub deployment.

### Data Analysis Key Findings
- Deduplicating across `3,000` streamed records (`1,500` `Apache-2.0` emails + `1,500` `MIT` SMS messages) and balancing `150` unique messages per `channel × verdict` stratum yields **`600` unique messages (`1,200` multimodal image-text examples)** split strictly at the unique-message level into **`960` Train (80%)**, **`120` Validation (10%)**, and **`120` Held-Out Test (10%)** pairs prior to fitting `95th-percentile` lexical risk scalers.
- Training-split lexical risk profiling confirms strong label-blind separation between `SPAM` and `LEGITIMATE` traffic across `Prize / Money`, `URL / Links`, `CTA / Phone`, `Urgency / Alert`, `Caps & '!'`, and `Digits / Symbols`, while neutral UI rendering guarantees zero visual chrome leakage.
- Side-by-side held-out test evaluation tracks **Accuracy, Spam Precision, Spam Recall, Spam F1, False Positive Rate, and Confusion Matrices** across unseen Email and SMS screenshots.

### Insights or Next Steps
- **Scale to Full Corpora**: Increase `EMAIL_STREAM_LIMIT` and `SMS_STREAM_LIMIT` to `5,500+` (`CURATED_SAMPLES_PER_STRATUM = 1000`) and run `num_train_epochs = 1` on a 24 GB+ CUDA GPU for production-grade anti-spam recall and ultra-low false positive rates.
- **Deploy Across Cloud & Edge**: Use [`SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit`](https://huggingface.co/SourinC/qwen2.5-vl-3b-email-sms-spam-rslora-4bit) for Python/GPU microservice batch screening and the Section 10 `GGUF` bundle (`Q4_K_M` + `mmproj-Q8_0`) for local desktop, CPU, or on-device privacy-preserving spam screening via `llama-server` or LM Studio.